<a href="https://colab.research.google.com/github/yongggquannn/bt4221-group-project/blob/us2.1-to-us2.5/bt4221_grp_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BT4221 Group Project: HDB Resale Price Prediction

This notebook implements an end-to-end pipeline for predicting HDB resale flat prices in Singapore using PySpark and LLM-guided agents.

**Pipeline Overview:**
1. **Auxiliary Dataset Extraction** — Hawker centres, shopping malls, supermarkets, MRT/LRT stations
2. **HDB Data Loading & Geocoding** — Download resale price data from Data.gov.sg and geocode addresses via OneMap
3. **LLM-Guided Data Cleaning** — Automated cleaning agent that profiles data and applies structured cleaning operations
4. **Exploratory Data Analysis** — Price trends, distributions, and feature correlations
5. **Feature Engineering & Preprocessing** — Time features, categorical encoding, interaction terms, multicollinearity analysis
6. **LangGraph ML Pipeline** — Orchestrated pipeline with feature engineering agent (distance, school quality, economic, demographic features), model training, and evaluation agent

**Prerequisites:** Google Colab (recommended) or a local environment with Java 17+ and PySpark.

**API Keys Required:** `GOV_DATA`, `ONEMAP_EMAIL`, `ONEMAP_PASSWORD`, `OPENAI_API_KEY` (set in Colab Secrets or `.env` file).

## Environment Setup

Install PySpark and Java runtime dependencies. The `apt-get` command targets Google Colab's Linux environment and is skipped on other platforms (e.g., macOS). Core PySpark modules are imported here.

In [9]:
!pip install pyspark -q
!apt-get install openjdk-17-jdk-headless -qq > /dev/null

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
The folder you are executing pip from can no longer be found.
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory


In [10]:
!pip install -q langgraph langchain langchain-openai openai python-dotenv beautifulsoup4 matplotlib seaborn

shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
The folder you are executing pip from can no longer be found.


In [11]:
# Mount Google Drive and navigate to the project folder
from google.colab import drive
drive.mount('/content/drive')

import os
# Update this path to where you placed the project folder in your Drive
os.chdir('/content/drive/MyDrive/bt4221-group-project')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Spark Session Initialization

Creates a local Spark session configured for the project with 8 shuffle partitions. Also defines `load_secret()`, a utility function that reads API keys from Google Colab Secrets, environment variables, or a `.env` file (checked in that priority order).

In [12]:
import os
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'

spark = (
    SparkSession.builder
    .appName("BT4221_HDB_Resale_Prices_Project")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.host", "localhost")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .getOrCreate()
)

print("Spark version:", spark.version)

spark.sparkContext.setLogLevel("WARN")

from pathlib import Path


def load_secret(name: str, *alt_names: str) -> str:
    """Colab Secrets, or environment / `.env` when running locally."""
    try:
        from google.colab import userdata

        val = userdata.get(name)
        if val:
            return val
    except ImportError:
        pass
    try:
        from dotenv import load_dotenv

        load_dotenv(Path.cwd() / ".env")
    except ImportError:
        pass
    for key in (name, *alt_names):
        v = os.environ.get(key)
        if v:
            return v
    raise ValueError(
        f"Missing secret {name!r}. "
        "On Colab: add it under Secrets. Locally: set the env var or add it to `.env`."
    )

Spark version: 4.0.2


# Section 1: Auxiliary Dataset Extraction

These cells extract location data for Singapore amenities (hawker centres, shopping malls, supermarkets, MRT/LRT stations) using LangGraph-orchestrated pipelines that combine web scraping, public APIs, and LLM-guided validation. Each pipeline saves its output as a CSV in the `dataset/` directory.

**Note:** These extraction cells are long-running and make external API calls (OneMap, OpenStreetMap, OpenAI). If the CSV files already exist in the `dataset/` directory, these cells will skip extraction automatically.

## 1a. Hawker Centre Extraction

Extracts hawker centre locations using a LangGraph pipeline:
- **Data source:** OpenStreetMap Overpass API (queries for amenity=hawker_centre in Singapore)
- **Geocoding:** OneMap API to convert addresses to lat/long coordinates
- **Validation:** LLM agent verifies data quality and completeness
- **Output:** `dataset/hawker_centre/hawker_centres.csv` (~100 rows with name, latitude, longitude)

In [13]:
import os
import json
import logging
from typing import TypedDict, Literal

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StringType, StructField, StructType, DoubleType, IntegerType,
)

from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

os.environ["SPARK_SUBMIT_OPTS"] = (
    "--add-opens java.base/javax.security.auth=ALL-UNNAMED "
    "--add-opens java.base/sun.nio.ch=ALL-UNNAMED "
    "--add-opens java.base/java.lang=ALL-UNNAMED "
    "--add-opens java.base/java.util=ALL-UNNAMED"
)

spark = (
    SparkSession.builder
    .appName("hawker_centre_extractor")
    .config("spark.driver.extraJavaOptions",
            "--add-opens java.base/javax.security.auth=ALL-UNNAMED "
            "--add-opens java.base/sun.nio.ch=ALL-UNNAMED "
            "--add-opens java.base/java.lang=ALL-UNNAMED "
            "--add-opens java.base/java.util=ALL-UNNAMED "
            "-Djava.security.manager=allow")
    .config("spark.executor.extraJavaOptions",
            "--add-opens java.base/javax.security.auth=ALL-UNNAMED "
            "-Djava.security.manager=allow")
    .getOrCreate()
)

llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0,
    api_key=load_secret("OPENAI_API_KEY"),
)

SCRIPT_DIR = os.path.join(os.getcwd(), "dataset/hawker_centre")
GEOJSON_PATH = os.path.join(SCRIPT_DIR, "HawkerCentresGEOJSON.geojson")

class PipelineState(TypedDict):
    geojson_path: str
    raw_features: list[dict]        # raw GeoJSON features
    extraction_plan: dict           # agent's plan on which fields to extract
    extracted_data: list[dict]      # flat dicts after field extraction
    validated_data: list[dict]      # after quality checks
    transformed_data: list[dict]    # final cleaned records
    quality_report: dict
    agent_decision: str
    agent_reasoning: str
    retry_count: int
    output_path: str


def plan_extraction(state: PipelineState) -> PipelineState:
    """LLM agent inspects the GeoJSON schema and decides which fields to extract."""
    logger.info("Node: plan_extraction")

    # Read the GeoJSON and grab a sample feature for schema inspection
    with open(state["geojson_path"], "r") as f:
        geojson = json.load(f)

    features = geojson.get("features", [])
    state["raw_features"] = features

    if not features:
        state["agent_decision"] = "abort"
        state["agent_reasoning"] = "GeoJSON contains no features."
        return state

    sample = features[0]
    sample_props = sample.get("properties", {})
    sample_geom = sample.get("geometry", {})

    response = llm.invoke([
        SystemMessage(content=(
            "You are a data engineering agent for a Singapore HDB resale price prediction project. "
            "You are given the schema of a GeoJSON file containing hawker centre locations. "
            "Decide which fields are useful for the ML pipeline. "
            "The target variable is HDB resale_price. Hawker centre proximity and size are "
            "known predictors of resale value.\n\n"
            "IMPORTANT:\n"
            "- Do NOT include fields whose sample value is null or empty.\n"
            "- Latitude and longitude are ALREADY extracted from geometry coordinates — "
            "do NOT add any lat/lng/lon/latitude/longitude/point/x/y coordinate fields.\n"
            "- Only use source_keys that exist in the sample properties keys provided.\n\n"
            "Respond ONLY with valid JSON:\n"
            "{\n"
            '  "decision": "proceed" | "abort",\n'
            '  "reasoning": "...",\n'
            '  "fields_to_extract": [\n'
            '    {"source_key": "NAME", "target_column": "hawker_name", "dtype": "string"},\n'
            "    ...\n"
            "  ],\n"
            '  "filter_status": "Existing"  // or null if no filter\n'
            "}"
        )),
        HumanMessage(content=(
            f"GeoJSON name: {geojson.get('name', 'unknown')}\n"
            f"Total features: {len(features)}\n"
            f"Sample geometry: {json.dumps(sample_geom)}\n"
            f"Sample properties keys: {list(sample_props.keys())}\n"
            f"Sample properties values: {json.dumps(sample_props, default=str)}"
        )),
    ])

    try:
        plan = json.loads(response.content)
    except json.JSONDecodeError:
        logger.warning("Agent returned non-JSON, using default extraction plan.")
        plan = _default_extraction_plan()

    # Validate plan has required keys, fall back to defaults if malformed
    if not isinstance(plan.get("fields_to_extract"), list) or len(plan["fields_to_extract"]) == 0:
        logger.warning("Agent plan missing fields_to_extract, using defaults.")
        plan = _default_extraction_plan()

    state["extraction_plan"] = plan
    state["agent_decision"] = plan.get("decision", "proceed")
    state["agent_reasoning"] = plan.get("reasoning", "")

    logger.info(
        "Agent plan: extract %d fields, filter_status=%s — %s",
        len(plan["fields_to_extract"]),
        plan.get("filter_status"),
        plan.get("reasoning", ""),
    )
    return state


def _default_extraction_plan() -> dict:
    """Fallback extraction plan if the LLM response is unusable."""
    return {
        "decision": "proceed",
        "reasoning": "Default plan: extract core hawker centre fields for price prediction.",
        "filter_status": "Existing",
        "fields_to_extract": [
            {"source_key": "NAME", "target_column": "hawker_name", "dtype": "string"},
            {"source_key": "ADDRESSPOSTALCODE", "target_column": "hawker_postal_code", "dtype": "string"},
            {"source_key": "ADDRESSSTREETNAME", "target_column": "hawker_street", "dtype": "string"},
            {"source_key": "ADDRESSBLOCKHOUSENUMBER", "target_column": "hawker_block", "dtype": "string"},
            {"source_key": "ADDRESS_MYENV", "target_column": "hawker_address", "dtype": "string"},
            {"source_key": "NUMBER_OF_COOKED_FOOD_STALLS", "target_column": "hawker_num_stalls", "dtype": "int"},
            {"source_key": "STATUS", "target_column": "hawker_status", "dtype": "string"},
            {"source_key": "EST_ORIGINAL_COMPLETION_DATE", "target_column": "hawker_est_completion", "dtype": "string"},
        ],
    }


def extract_data(state: PipelineState) -> PipelineState:
    """Extract flat records from raw GeoJSON features based on the agent's plan."""
    logger.info("Node: extract_data")

    plan = state["extraction_plan"]
    field_map = plan["fields_to_extract"]
    filter_status = plan.get("filter_status")

    records = []
    skipped = 0

    for feature in state["raw_features"]:
        props = feature.get("properties", {})
        geom = feature.get("geometry", {})
        coords = geom.get("coordinates", [None, None])

        # Optional status filter
        if filter_status and props.get("STATUS", "") != filter_status:
            skipped += 1
            continue

        row = {
            "hawker_lat": coords[1] if len(coords) > 1 else None,
            "hawker_lng": coords[0] if len(coords) > 0 else None,
        }

        for fm in field_map:
            src = fm["source_key"]
            tgt = fm["target_column"]
            val = props.get(src)

            # Type coercion
            if fm.get("dtype") == "int" and val is not None:
                try:
                    val = int(val)
                except (ValueError, TypeError):
                    val = None
            elif fm.get("dtype") == "double" and val is not None:
                try:
                    val = float(val)
                except (ValueError, TypeError):
                    val = None

            row[tgt] = val

        records.append(row)

    state["extracted_data"] = records
    state["quality_report"] = {
        "total_features": len(state["raw_features"]),
        "extracted": len(records),
        "filtered_out": skipped,
        "filter_status": filter_status,
        "null_lat_count": sum(1 for r in records if r.get("hawker_lat") is None),
        "null_name_count": sum(1 for r in records if r.get("hawker_name") is None),
    }

    logger.info(
        "Extracted %d records (%d filtered out by status='%s')",
        len(records), skipped, filter_status,
    )
    return state


def validate_extraction(state: PipelineState) -> PipelineState:
    """LLM agent reviews extraction quality and decides next step."""
    logger.info("Node: validate_extraction")

    qr = state["quality_report"]

    response = llm.invoke([
        SystemMessage(content=(
            "You are a data quality agent. Evaluate the hawker centre extraction report. "
            "Respond ONLY with valid JSON:\n"
            '{"decision": "proceed"|"retry"|"abort", "reasoning": "..."}\n'
            "Rules:\n"
            "- If extracted >= 80% of total features (after valid filtering): proceed\n"
            "- If null_lat_count > 10% of extracted: retry (max 2 retries)\n"
            "- If extracted == 0: abort"
        )),
        HumanMessage(content=(
            f"Quality report: {json.dumps(qr)}. "
            f"Retry count: {state['retry_count']}. Max retries: 2."
        )),
    ])

    try:
        agent_output = json.loads(response.content)
    except json.JSONDecodeError:
        # Fallback rule-based decision
        if qr["extracted"] == 0:
            agent_output = {"decision": "abort", "reasoning": "No records extracted."}
        elif qr["null_lat_count"] / max(qr["extracted"], 1) > 0.1:
            agent_output = {"decision": "retry", "reasoning": "Too many null coordinates."}
        else:
            agent_output = {"decision": "proceed", "reasoning": "Fallback: quality looks acceptable."}

    state["agent_decision"] = agent_output["decision"]
    state["agent_reasoning"] = agent_output["reasoning"]

    if agent_output["decision"] == "retry":
        state["retry_count"] += 1

    logger.info("Agent QA Decision: %s — %s", agent_output["decision"], agent_output["reasoning"])
    return state


def transform_data(state: PipelineState) -> PipelineState:
    """Use PySpark to clean and finalize the extracted hawker centre data."""
    logger.info("Node: transform_data")

    records = state["extracted_data"]
    if not records:
        state["transformed_data"] = []
        return state

    # Infer columns from extraction plan + lat/lng
    columns = ["hawker_lat", "hawker_lng"] + [
        fm["target_column"] for fm in state["extraction_plan"]["fields_to_extract"]
    ]

    # Build PySpark schema dynamically
    type_map = {"string": StringType(), "int": IntegerType(), "double": DoubleType()}
    fields = [
        StructField("hawker_lat", DoubleType(), True),
        StructField("hawker_lng", DoubleType(), True),
    ]
    for fm in state["extraction_plan"]["fields_to_extract"]:
        spark_type = type_map.get(fm.get("dtype", "string"), StringType())
        fields.append(StructField(fm["target_column"], spark_type, True))

    schema = StructType(fields)

    # Create DataFrame
    rows = [{col: r.get(col) for col in columns} for r in records]
    df = spark.createDataFrame(rows, schema=schema)

    # Drop duplicates
    df = df.dropDuplicates()

    # Drop rows with null coordinates
    df = df.filter(F.col("hawker_lat").isNotNull() & F.col("hawker_lng").isNotNull())

    # Drop rows with null name
    if "hawker_name" in df.columns:
        df = df.filter(F.col("hawker_name").isNotNull())

    # Round coordinates to 8 decimal places for consistency
    df = df.withColumn("hawker_lat", F.round(F.col("hawker_lat"), 8))
    df = df.withColumn("hawker_lng", F.round(F.col("hawker_lng"), 8))

    state["transformed_data"] = [row.asDict() for row in df.collect()]
    logger.info("Transformed %d hawker centre records", len(state["transformed_data"]))
    return state

def decide_output(state: PipelineState) -> PipelineState:
    """Write final hawker centre data to CSV."""
    logger.info("Node: decide_output")

    output_dir = SCRIPT_DIR
    os.makedirs(output_dir, exist_ok=True)
    path = os.path.join(output_dir, "hawker_centres.csv")

    # Build explicit schema to avoid inference failures on all-null columns
    type_map = {"string": StringType(), "int": IntegerType(), "double": DoubleType()}
    fields = [
        StructField("hawker_lat", DoubleType(), True),
        StructField("hawker_lng", DoubleType(), True),
    ]
    for fm in state["extraction_plan"]["fields_to_extract"]:
        spark_type = type_map.get(fm.get("dtype", "string"), StringType())
        fields.append(StructField(fm["target_column"], spark_type, True))
    schema = StructType(fields)

    df = spark.createDataFrame(state["transformed_data"], schema=schema)

    # Reorder columns: name first, then location, then metadata
    preferred_order = [
        "hawker_name", "hawker_lat", "hawker_lng", "hawker_postal_code",
        "hawker_street", "hawker_block", "hawker_address",
        "hawker_num_stalls", "hawker_status", "hawker_est_completion",
    ]
    existing_cols = [c for c in preferred_order if c in df.columns]
    remaining_cols = [c for c in df.columns if c not in existing_cols]
    df = df.select(existing_cols + remaining_cols)

    df.toPandas().to_csv(path, index=False)

    state["output_path"] = path
    state["agent_reasoning"] = f"CSV output written to {path}"
    logger.info("Output written to %s (%d rows)", path, len(state["transformed_data"]))
    return state

def route_after_plan(state: PipelineState) -> Literal["extract_data", "__end__"]:
    if state["agent_decision"] == "abort":
        logger.warning("Agent aborted at planning: %s", state["agent_reasoning"])
        return END
    return "extract_data"


def route_after_validation(
    state: PipelineState,
) -> Literal["transform_data", "extract_data", "__end__"]:
    if state["agent_decision"] == "proceed":
        return "transform_data"
    elif state["agent_decision"] == "retry" and state["retry_count"] <= 2:
        return "extract_data"
    else:
        logger.warning("Agent aborted after validation: %s", state["agent_reasoning"])
        return END


# Build LangGraph Pipeline

workflow = StateGraph(PipelineState)

workflow.add_node("plan_extraction", plan_extraction)
workflow.add_node("extract_data", extract_data)
workflow.add_node("validate_extraction", validate_extraction)
workflow.add_node("transform_data", transform_data)
workflow.add_node("decide_output", decide_output)

workflow.set_entry_point("plan_extraction")
workflow.add_conditional_edges("plan_extraction", route_after_plan)
workflow.add_edge("extract_data", "validate_extraction")
workflow.add_conditional_edges("validate_extraction", route_after_validation)
workflow.add_edge("transform_data", "decide_output")
workflow.add_edge("decide_output", END)

app = workflow.compile()

## 1b. Shopping Mall Extraction

Extracts shopping mall locations using a LangGraph pipeline:
- **Data source:** Web scraping from Wikipedia for mall names
- **Geocoding:** OneMap API to obtain lat/long coordinates for each mall
- **Validation:** LLM agent handles deduplication and data quality checks
- **Output:** `dataset/shopping_mall/shopping_malls.csv`

In [14]:
import os
import json
import requests
import logging
import time
from typing import TypedDict, Literal, Optional
from bs4 import BeautifulSoup
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, StructField, StructType, DoubleType

from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

os.environ["SPARK_SUBMIT_OPTS"] = (
    "--add-opens java.base/javax.security.auth=ALL-UNNAMED "
    "--add-opens java.base/sun.nio.ch=ALL-UNNAMED "
    "--add-opens java.base/java.lang=ALL-UNNAMED "
    "--add-opens java.base/java.util=ALL-UNNAMED"
)

spark = (
    SparkSession.builder
    .appName("shopping_mall_extractor")
    .config("spark.driver.extraJavaOptions",
            "--add-opens java.base/javax.security.auth=ALL-UNNAMED "
            "--add-opens java.base/sun.nio.ch=ALL-UNNAMED "
            "--add-opens java.base/java.lang=ALL-UNNAMED "
            "--add-opens java.base/java.util=ALL-UNNAMED "
            "-Djava.security.manager=allow")
    .config("spark.executor.extraJavaOptions",
            "--add-opens java.base/javax.security.auth=ALL-UNNAMED "
            "-Djava.security.manager=allow")
    .getOrCreate()
)

llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0,
    api_key=load_secret("OPENAI_API_KEY"),
)


class PipelineState(TypedDict):
    mall_names: list[str]
    raw_data: list[dict]
    validated_data: list[dict]
    transformed_data: list[dict]
    quality_report: dict
    agent_decision: str
    retry_count: int
    agent_reasoning: str
    output_path: str


def scrape_mall_names() -> list[str]:
    """Scrape shopping mall names from the Wikipedia list page using BeautifulSoup."""
    url = "https://en.wikipedia.org/wiki/List_of_shopping_malls_in_Singapore"
    headers = {"User-Agent": "Mozilla/5.0 (compatible; BT4221-Bot/1.0)"}

    resp = requests.get(url, headers=headers, timeout=15)
    resp.raise_for_status()

    soup = BeautifulSoup(resp.text, "lxml")

    # Target region headings: Central, East, North, North-East, West
    target_regions = {"Central", "East", "North", "North-East", "West"}
    all_malls = []

    content_div = soup.find("div", class_="mw-parser-output")
    if not content_div:
        logger.error("Could not find mw-parser-output div on Wikipedia page")
        return []

    # Iterate through direct children of the content div sequentially
    current_region = None
    for child in content_div.children:
        if not hasattr(child, "name") or child.name is None:
            continue

        # Check for heading div (region header)
        if child.name == "div" and "mw-heading2" in (child.get("class") or []):
            h2 = child.find("h2")
            if h2:
                region = h2.get("id") or h2.get_text(strip=True)
                if region in target_regions:
                    current_region = region
                else:
                    current_region = None
            continue

        # If we're in a target region, look for div-col with mall lists
        if current_region and child.name == "div" and "div-col" in (child.get("class") or []):
            for li in child.find_all("li"):
                mall_name = li.get_text(strip=True)
                if mall_name:
                    all_malls.append(mall_name)

    logger.info("Scraped %d mall entries from Wikipedia across %d regions",
                len(all_malls), len(target_regions))
    return all_malls


def plan_extraction(state: PipelineState) -> PipelineState:
    """Scrape Wikipedia for all shopping mall names in Singapore, then validate with agent."""
    # Web scrape mall names from Wikipedia
    try:
        scraped_malls = scrape_mall_names()
    except Exception as e:
        logger.error("Failed to scrape Wikipedia: %s", e)
        state["agent_decision"] = "abort"
        state["agent_reasoning"] = f"Wikipedia scrape failed: {e}"
        return state

    # Deduplicate by normalised name (case-insensitive)
    seen = set()
    unique_malls = []
    for m in scraped_malls:
        key = m.strip().lower()
        if key not in seen:
            seen.add(key)
            unique_malls.append(m.strip())

    # Agent validates the scraped list
    response = llm.invoke([
        SystemMessage(content=(
            "You are a data engineering agent. Evaluate the shopping mall list scraped from "
            "Wikipedia (https://en.wikipedia.org/wiki/List_of_shopping_malls_in_Singapore) and "
            "respond ONLY with valid JSON: "
            '{"decision": "proceed"|"abort", "reasoning": "...", "mall_count": <int>}'
        )),
        HumanMessage(content=(
            f"I scraped {len(unique_malls)} unique shopping malls from Wikipedia. "
            f"Sample: {unique_malls[:5]}...{unique_malls[-5:]}. "
            f"Should I proceed with OneMap geocoding extraction?"
        )),
    ])

    try:
        agent_output = json.loads(response.content)
    except json.JSONDecodeError:
        agent_output = {"decision": "proceed", "reasoning": "Defaulting to proceed", "mall_count": len(unique_malls)}

    # Override abort if count is within expected range
    if agent_output["decision"] == "abort" and 20 <= len(unique_malls) <= 500:
        logger.warning("Agent suggested abort but mall count looks valid (%d). Overriding to proceed.", len(unique_malls))
        agent_output["decision"] = "proceed"
        agent_output["reasoning"] = f"Overridden: {agent_output['reasoning']}"

    logger.info("Scraped %d total unique malls from Wikipedia", len(unique_malls))
    logger.info("Agent Plan Decision: %s — %s", agent_output["decision"], agent_output["reasoning"])

    state["mall_names"] = unique_malls
    state["agent_decision"] = agent_output["decision"]
    state["agent_reasoning"] = agent_output["reasoning"]
    return state


def _onemap_search(search_val: str, max_retries: int = 3) -> Optional[dict]:
    """Call OneMap search API with retry logic for rate limiting."""
    for attempt in range(max_retries):
        try:
            url = (
                f"https://www.onemap.gov.sg/api/common/elastic/search"
                f"?searchVal={search_val}&returnGeom=Y&getAddrDetails=Y&pageNum=1"
            )
            resp = requests.get(url, timeout=10)

            if resp.status_code == 429 or resp.status_code >= 500:
                wait = 2 ** attempt
                logger.info("Rate limited on '%s', retrying in %ds...", search_val, wait)
                time.sleep(wait)
                continue

            if resp.status_code != 200:
                return None

            ctype = resp.headers.get("Content-Type", "")
            if "application/json" not in ctype.lower():
                # Empty/non-JSON response often means rate limiting
                wait = 2 ** attempt
                logger.info("Non-JSON response for '%s', retrying in %ds...", search_val, wait)
                time.sleep(wait)
                continue

            return resp.json()

        except Exception as e:
            logger.warning("Request error for '%s' (attempt %d): %s", search_val, attempt + 1, e)
            time.sleep(2 ** attempt)

    return None


def extract_data(state: PipelineState) -> PipelineState:
    """Search OneMap using shopping mall names for geocoding results."""
    mall_data = []
    failed_names = []

    # Alias map: Wikipedia name -> list of alternative OneMap search terms
    alias_map = {
        "Holland Village Shopping Mall": ["Holland Village"],
        "Shaw House and Centre": ["Shaw House", "Shaw Centre"],
        "Novena Square Shopping Mall": ["Novena Square"],
        "i12 Katong": ["112 Katong", "I12 KATONG"],
        "Yew Tee Point": ["YEW TEE POINT", "Yew Tee"],
        "SingPostCentre": ["SingPost Centre", "Singapore Post Centre"],
        "Lot 1": ["Lot One Shoppers Mall", "LOT 1"],
    }

    for idx, name in enumerate(state["mall_names"]):
        # Build search term list: start with aliases if available, then the name itself
        if name in alias_map:
            search_terms = alias_map[name] + [name]
        else:
            search_terms = [name]

        # Add generic fallback variations
        if not any(kw in name.lower() for kw in ["mall", "shopping", "centre", "center", "plaza"]):
            search_terms.append(f"{name} Mall")
            search_terms.append(f"{name} Shopping Centre")

        found = False
        for search_val in search_terms:
            data = _onemap_search(search_val)
            if not data:
                continue

            if "results" in data and isinstance(data["results"], list) and data["results"]:
                best_result = None

                for result in data["results"]:
                    building = result.get("BUILDING", "").upper()
                    search_val_upper = result.get("SEARCHVAL", "").upper()
                    name_upper = name.upper()

                    # Priority 1: mall name appears in BUILDING or SEARCHVAL
                    if name_upper in building or name_upper in search_val_upper:
                        best_result = result
                        break

                    # Priority 2: keyword match (MALL, SHOPPING, PLAZA, etc.)
                    if not best_result and any(
                        kw in building or kw in search_val_upper
                        for kw in ["MALL", "SHOPPING", "PLAZA", "CENTRE", "CENTER",
                                   "POINT", "CITY", "SQUARE", "PLACE", "JUNCTION",
                                   "HUB", "LINK", "VILLAGE", "GALLERY"]
                    ):
                        best_result = result

                # Priority 3: fallback to the first result if no keyword match
                if not best_result:
                    best_result = data["results"][0]

                best_result["MALL_NAME"] = name
                mall_data.append(best_result)
                found = True

            if found:
                break

        if not found:
            logger.warning("Failed to find mall: %s", name)
            failed_names.append(name)

        # Throttle: 0.5s between requests to avoid rate limiting
        time.sleep(0.5)

        if (idx + 1) % 20 == 0:
            logger.info("Progress: %d / %d malls searched", idx + 1, len(state["mall_names"]))

    state["raw_data"] = mall_data
    state["quality_report"] = {
        "total_malls": len(state["mall_names"]),
        "extracted": len(mall_data),
        "failed": len(failed_names),
        "failed_names": failed_names[:20],
        "success_rate": round(len(mall_data) / max(len(state["mall_names"]), 1) * 100, 2),
    }
    logger.info("Extracted %d / %d malls", len(mall_data), len(state["mall_names"]))
    return state


def validate_extraction(state: PipelineState) -> PipelineState:
    """Agent reviews extraction quality and decides whether to proceed, retry, or abort."""
    qr = state["quality_report"]

    response = llm.invoke([
        SystemMessage(content=(
            "You are a data quality agent. Evaluate the extraction results and respond "
            "ONLY with valid JSON: "
            '{"decision": "proceed"|"retry"|"abort", "reasoning": "..."}'
            "\nRules: success_rate >= 80% → proceed; 50-80% → retry (max 2); <50% → abort."
        )),
        HumanMessage(content=(
            f"Extraction report: {json.dumps(qr)}. "
            f"Current retry count: {state['retry_count']}. "
            f"Max retries: 2. What should we do?"
        )),
    ])

    try:
        agent_output = json.loads(response.content)
    except json.JSONDecodeError:
        agent_output = {
            "decision": "proceed" if qr["success_rate"] >= 80 else "abort",
            "reasoning": "Fallback rule-based decision",
        }

    state["agent_decision"] = agent_output["decision"]
    state["agent_reasoning"] = agent_output["reasoning"]

    if agent_output["decision"] == "retry":
        state["retry_count"] += 1

    logger.info("Agent QA Decision: %s — %s", agent_output["decision"], agent_output["reasoning"])
    return state


def transform_data(state: PipelineState) -> PipelineState:
    """Transform raw OneMap results into a clean schema using PySpark."""
    schema = StructType([
        StructField("SEARCHVAL", StringType(), True),
        StructField("BLK_NO", StringType(), True),
        StructField("ROAD_NAME", StringType(), True),
        StructField("BUILDING", StringType(), True),
        StructField("ADDRESS", StringType(), True),
        StructField("POSTAL", StringType(), True),
        StructField("X", StringType(), True),
        StructField("Y", StringType(), True),
        StructField("LATITUDE", StringType(), True),
        StructField("LONGITUDE", StringType(), True),
        StructField("MALL_NAME", StringType(), True),
    ])

    df = spark.createDataFrame(state["raw_data"], schema=schema)
    df = df.dropDuplicates()
    df = df.drop("SEARCHVAL", "BLK_NO", "X", "Y")
    df = (
        df.withColumnRenamed("ROAD_NAME", "roadName")
          .withColumnRenamed("BUILDING", "building")
          .withColumnRenamed("ADDRESS", "address")
          .withColumnRenamed("POSTAL", "postalCode")
          .withColumnRenamed("LATITUDE", "latitude")
          .withColumnRenamed("LONGITUDE", "longitude")
          .withColumnRenamed("MALL_NAME", "mallName")
    )
    df = df.withColumn("latitude", F.col("latitude").cast(DoubleType()))
    df = df.withColumn("longitude", F.col("longitude").cast(DoubleType()))

    state["transformed_data"] = [row.asDict() for row in df.collect()]
    logger.info("Transformed %d mall records", df.count())
    return state


def decide_output(state: PipelineState) -> PipelineState:
    """Write cleaned shopping mall data to same directory as this script."""
    output_dir = os.path.join(os.getcwd(), "dataset/shopping_mall")
    os.makedirs(output_dir, exist_ok=True)

    path = os.path.join(output_dir, "shopping_malls.csv")

    df = spark.createDataFrame(state["transformed_data"])
    df.toPandas().to_csv(path, index=False)

    state["output_path"] = path
    state["agent_reasoning"] = "CSV output to dataset/shopping_mall."
    logger.info("Output written to %s", path)
    return state


def route_after_plan(state: PipelineState) -> Literal["extract_data", "__end__"]:
    if state["agent_decision"] == "abort":
        logger.warning("Agent aborted pipeline: %s", state["agent_reasoning"])
        return END
    return "extract_data"


def route_after_validation(state: PipelineState) -> Literal["transform_data", "extract_data", "__end__"]:
    if state["agent_decision"] == "proceed":
        return "transform_data"
    elif state["agent_decision"] == "retry" and state["retry_count"] <= 2:
        return "extract_data"
    else:
        logger.warning("Agent aborted after validation: %s", state["agent_reasoning"])
        return END


# Build LangGraph workflow
workflow = StateGraph(PipelineState)

workflow.add_node("plan_extraction", plan_extraction)
workflow.add_node("extract_data", extract_data)
workflow.add_node("validate_extraction", validate_extraction)
workflow.add_node("transform_data", transform_data)
workflow.add_node("decide_output", decide_output)

workflow.set_entry_point("plan_extraction")
workflow.add_conditional_edges("plan_extraction", route_after_plan)
workflow.add_edge("extract_data", "validate_extraction")
workflow.add_conditional_edges("validate_extraction", route_after_validation)
workflow.add_edge("transform_data", "decide_output")
workflow.add_edge("decide_output", END)

app = workflow.compile()

## 1c. Supermarket Extraction

Extracts supermarket locations using a LangGraph pipeline:
- **Data source:** Data.gov.sg GeoJSON API for licensed supermarket locations
- **Parsing:** Extracts coordinates and names from GeoJSON features, parses HTML descriptions
- **Validation:** LLM agent validates completeness
- **Output:** `dataset/supermarket/supermarkets.csv` (~400 rows)

In [17]:
import os
import re
import json
import logging
from typing import TypedDict, Literal

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StringType, StructField, StructType, DoubleType, IntegerType,
)

from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

os.environ["SPARK_SUBMIT_OPTS"] = (
    "--add-opens java.base/javax.security.auth=ALL-UNNAMED "
    "--add-opens java.base/sun.nio.ch=ALL-UNNAMED "
    "--add-opens java.base/java.lang=ALL-UNNAMED "
    "--add-opens java.base/java.util=ALL-UNNAMED"
)

spark = (
    SparkSession.builder
    .appName("supermarket_extractor")
    .config("spark.driver.extraJavaOptions",
            "--add-opens java.base/javax.security.auth=ALL-UNNAMED "
            "--add-opens java.base/sun.nio.ch=ALL-UNNAMED "
            "--add-opens java.base/java.lang=ALL-UNNAMED "
            "--add-opens java.base/java.util=ALL-UNNAMED "
            "-Djava.security.manager=allow")
    .config("spark.executor.extraJavaOptions",
            "--add-opens java.base/javax.security.auth=ALL-UNNAMED "
            "-Djava.security.manager=allow")
    .getOrCreate()
)

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    api_key=load_secret("OPENAI_API_KEY"),
)

SCRIPT_DIR = os.path.join(os.getcwd(), "dataset/supermarket")
GEOJSON_PATH = os.path.join(SCRIPT_DIR, "SupermarketsGEOJSON.geojson")


# ---------------------------------------------------------------------------
# Utility: parse HTML description table into a flat dict
# ---------------------------------------------------------------------------
def _parse_html_description(html: str) -> dict:
    """Extract key-value pairs from the KML-style HTML table in Description."""
    pairs: dict[str, str] = {}
    # Each row looks like: <th>KEY</th> <td>VALUE</td>
    for match in re.finditer(
        r"<th>\s*([^<]+?)\s*</th>\s*<td>\s*([^<]*?)\s*</td>",
        html,
        re.IGNORECASE,
    ):
        key = match.group(1).strip()
        val = match.group(2).strip()
        if key.lower() != "attributes":  # skip the header row
            pairs[key] = val
    return pairs


# ---------------------------------------------------------------------------
# State definition
# ---------------------------------------------------------------------------
class PipelineState(TypedDict):
    geojson_path: str
    raw_features: list[dict]        # raw GeoJSON features
    extraction_plan: dict           # agent's plan on which fields to extract
    extracted_data: list[dict]      # flat dicts after field extraction
    validated_data: list[dict]      # after quality checks
    transformed_data: list[dict]    # final cleaned records
    quality_report: dict
    agent_decision: str
    agent_reasoning: str
    retry_count: int
    output_path: str


# ---------------------------------------------------------------------------
# Pipeline nodes
# ---------------------------------------------------------------------------
def plan_extraction(state: PipelineState) -> PipelineState:
    """LLM agent inspects the GeoJSON schema and decides which fields to extract."""
    logger.info("Node: plan_extraction")

    with open(state["geojson_path"], "r") as f:
        geojson = json.load(f)

    features = geojson.get("features", [])
    state["raw_features"] = features

    if not features:
        state["agent_decision"] = "abort"
        state["agent_reasoning"] = "GeoJSON contains no features."
        return state

    # Parse a sample feature so the LLM can inspect available keys
    sample = features[0]
    sample_geom = sample.get("geometry", {})
    sample_parsed = _parse_html_description(
        sample.get("properties", {}).get("Description", "")
    )

    response = llm.invoke([
        SystemMessage(content=(
            "You are a data engineering agent for a Singapore HDB resale price prediction project. "
            "You are given the schema of a GeoJSON file containing supermarket locations. "
            "The raw properties are embedded in an HTML table inside the 'Description' field "
            "and have already been parsed into key-value pairs for you.\n\n"
            "Decide which fields are useful for the ML pipeline. "
            "The target variable is HDB resale_price. Supermarket proximity is a "
            "known predictor of resale value.\n\n"
            "IMPORTANT:\n"
            "- Do NOT include fields whose sample value is null or empty.\n"
            "- Latitude and longitude are ALREADY extracted from geometry coordinates — "
            "do NOT add any lat/lng/lon/latitude/longitude fields.\n"
            "- Only use source_keys that appear in the parsed description keys provided.\n\n"
            "Respond ONLY with valid JSON:\n"
            "{\n"
            '  "decision": "proceed" | "abort",\n'
            '  "reasoning": "...",\n'
            '  "fields_to_extract": [\n'
            '    {"source_key": "LIC_NAME", "target_column": "supermarket_name", "dtype": "string"},\n'
            "    ...\n"
            "  ]\n"
            "}"
        )),
        HumanMessage(content=(
            f"Total features: {len(features)}\n"
            f"Sample geometry: {json.dumps(sample_geom)}\n"
            f"Parsed description keys: {list(sample_parsed.keys())}\n"
            f"Parsed description values: {json.dumps(sample_parsed, default=str)}"
        )),
    ])

    try:
        plan = json.loads(response.content)
    except json.JSONDecodeError:
        logger.warning("Agent returned non-JSON, using default extraction plan.")
        plan = _default_extraction_plan()

    if not isinstance(plan.get("fields_to_extract"), list) or len(plan["fields_to_extract"]) == 0:
        logger.warning("Agent plan missing fields_to_extract, using defaults.")
        plan = _default_extraction_plan()

    state["extraction_plan"] = plan
    state["agent_decision"] = plan.get("decision", "proceed")
    state["agent_reasoning"] = plan.get("reasoning", "")

    logger.info(
        "Agent plan: extract %d fields — %s",
        len(plan["fields_to_extract"]),
        plan.get("reasoning", ""),
    )
    return state


def _default_extraction_plan() -> dict:
    """Fallback extraction plan if the LLM response is unusable."""
    return {
        "decision": "proceed",
        "reasoning": "Default plan: extract core supermarket fields for price prediction.",
        "fields_to_extract": [
            {"source_key": "LIC_NAME", "target_column": "supermarket_name", "dtype": "string"},
            {"source_key": "BLK_HOUSE", "target_column": "supermarket_block", "dtype": "string"},
            {"source_key": "STR_NAME", "target_column": "supermarket_street", "dtype": "string"},
            {"source_key": "UNIT_NO", "target_column": "supermarket_unit", "dtype": "string"},
            {"source_key": "POSTCODE", "target_column": "supermarket_postal_code", "dtype": "string"},
            {"source_key": "LIC_NO", "target_column": "supermarket_licence_no", "dtype": "string"},
        ],
    }


def extract_data(state: PipelineState) -> PipelineState:
    """Extract flat records from raw GeoJSON features based on the agent's plan."""
    logger.info("Node: extract_data")

    plan = state["extraction_plan"]
    field_map = plan["fields_to_extract"]

    records: list[dict] = []

    for feature in state["raw_features"]:
        props = feature.get("properties", {})
        geom = feature.get("geometry", {})
        coords = geom.get("coordinates", [None, None])

        # Parse the HTML description into a flat dict
        parsed = _parse_html_description(props.get("Description", ""))

        row: dict = {
            "supermarket_lat": coords[1] if len(coords) > 1 else None,
            "supermarket_lng": coords[0] if len(coords) > 0 else None,
        }

        for fm in field_map:
            # Skip if the agent plan duplicates the hardcoded lat/lng columns
            if fm["target_column"] in ("supermarket_lat", "supermarket_lng"):
                continue
            src = fm["source_key"]
            tgt = fm["target_column"]
            val = parsed.get(src)

            # Type coercion
            if fm.get("dtype") == "int" and val is not None:
                try:
                    val = int(val)
                except (ValueError, TypeError):
                    val = None
            elif fm.get("dtype") == "double" and val is not None:
                try:
                    val = float(val)
                except (ValueError, TypeError):
                    val = None

            row[tgt] = val

        records.append(row)

    state["extracted_data"] = records
    state["quality_report"] = {
        "total_features": len(state["raw_features"]),
        "extracted": len(records),
        "null_lat_count": sum(1 for r in records if r.get("supermarket_lat") is None),
        "null_name_count": sum(1 for r in records if r.get("supermarket_name") is None),
    }

    logger.info("Extracted %d supermarket records", len(records))
    return state


def validate_extraction(state: PipelineState) -> PipelineState:
    """LLM agent reviews extraction quality and decides next step."""
    logger.info("Node: validate_extraction")

    qr = state["quality_report"]

    response = llm.invoke([
        SystemMessage(content=(
            "You are a data quality agent. Evaluate the supermarket extraction report. "
            "Respond ONLY with valid JSON:\n"
            '{"decision": "proceed"|"retry"|"abort", "reasoning": "..."}\n'
            "Rules:\n"
            "- If extracted >= 80% of total features: proceed\n"
            "- If null_lat_count > 10% of extracted: retry (max 2 retries)\n"
            "- If extracted == 0: abort"
        )),
        HumanMessage(content=(
            f"Quality report: {json.dumps(qr)}. "
            f"Retry count: {state['retry_count']}. Max retries: 2."
        )),
    ])

    try:
        agent_output = json.loads(response.content)
    except json.JSONDecodeError:
        if qr["extracted"] == 0:
            agent_output = {"decision": "abort", "reasoning": "No records extracted."}
        elif qr["null_lat_count"] / max(qr["extracted"], 1) > 0.1:
            agent_output = {"decision": "retry", "reasoning": "Too many null coordinates."}
        else:
            agent_output = {"decision": "proceed", "reasoning": "Fallback: quality looks acceptable."}

    state["agent_decision"] = agent_output["decision"]
    state["agent_reasoning"] = agent_output["reasoning"]

    if agent_output["decision"] == "retry":
        state["retry_count"] += 1

    logger.info("Agent QA Decision: %s — %s", agent_output["decision"], agent_output["reasoning"])
    return state


def transform_data(state: PipelineState) -> PipelineState:
    """Use PySpark to clean and finalize the extracted supermarket data."""
    logger.info("Node: transform_data")

    records = state["extracted_data"]
    if not records:
        state["transformed_data"] = []
        return state

    # Infer columns from extraction plan + lat/lng (exclude duplicates)
    plan_fields = [
        fm for fm in state["extraction_plan"]["fields_to_extract"]
        if fm["target_column"] not in ("supermarket_lat", "supermarket_lng")
    ]
    columns = ["supermarket_lat", "supermarket_lng"] + [
        fm["target_column"] for fm in plan_fields
    ]

    # Build PySpark schema dynamically
    type_map = {"string": StringType(), "int": IntegerType(), "double": DoubleType()}
    fields = [
        StructField("supermarket_lat", DoubleType(), True),
        StructField("supermarket_lng", DoubleType(), True),
    ]
    for fm in plan_fields:
        spark_type = type_map.get(fm.get("dtype", "string"), StringType())
        fields.append(StructField(fm["target_column"], spark_type, True))

    schema = StructType(fields)

    rows = [{col: r.get(col) for col in columns} for r in records]
    df = spark.createDataFrame(rows, schema=schema)

    # Drop duplicates
    df = df.dropDuplicates()

    # Drop rows with null coordinates
    df = df.filter(F.col("supermarket_lat").isNotNull() & F.col("supermarket_lng").isNotNull())

    # Drop rows with null name
    if "supermarket_name" in df.columns:
        df = df.filter(F.col("supermarket_name").isNotNull())

    # Round coordinates to 8 decimal places for consistency
    df = df.withColumn("supermarket_lat", F.round(F.col("supermarket_lat"), 8))
    df = df.withColumn("supermarket_lng", F.round(F.col("supermarket_lng"), 8))

    state["transformed_data"] = [row.asDict() for row in df.collect()]
    logger.info("Transformed %d supermarket records", len(state["transformed_data"]))
    return state


def decide_output(state: PipelineState) -> PipelineState:
    """Write final supermarket data to CSV."""
    logger.info("Node: decide_output")

    output_dir = SCRIPT_DIR
    os.makedirs(output_dir, exist_ok=True)
    path = os.path.join(output_dir, "supermarkets.csv")

    # Build explicit schema (exclude duplicates of lat/lng)
    plan_fields = [
        fm for fm in state["extraction_plan"]["fields_to_extract"]
        if fm["target_column"] not in ("supermarket_lat", "supermarket_lng")
    ]
    type_map = {"string": StringType(), "int": IntegerType(), "double": DoubleType()}
    fields = [
        StructField("supermarket_lat", DoubleType(), True),
        StructField("supermarket_lng", DoubleType(), True),
    ]
    for fm in plan_fields:
        spark_type = type_map.get(fm.get("dtype", "string"), StringType())
        fields.append(StructField(fm["target_column"], spark_type, True))
    schema = StructType(fields)

    df = spark.createDataFrame(state["transformed_data"], schema=schema)

    # Reorder columns: name first, then location, then metadata
    preferred_order = [
        "supermarket_name", "supermarket_lat", "supermarket_lng",
        "supermarket_postal_code", "supermarket_street", "supermarket_block",
        "supermarket_unit", "supermarket_licence_no",
    ]
    existing_cols = [c for c in preferred_order if c in df.columns]
    remaining_cols = [c for c in df.columns if c not in existing_cols]
    df = df.select(existing_cols + remaining_cols)

    df.toPandas().to_csv(path, index=False)

    state["output_path"] = path
    state["agent_reasoning"] = f"CSV output written to {path}"
    logger.info("Output written to %s (%d rows)", path, len(state["transformed_data"]))
    return state


# ---------------------------------------------------------------------------
# Routing functions
# ---------------------------------------------------------------------------
def route_after_plan(state: PipelineState) -> Literal["extract_data", "__end__"]:
    if state["agent_decision"] == "abort":
        logger.warning("Agent aborted at planning: %s", state["agent_reasoning"])
        return END
    return "extract_data"


def route_after_validation(
    state: PipelineState,
) -> Literal["transform_data", "extract_data", "__end__"]:
    if state["agent_decision"] == "proceed":
        return "transform_data"
    elif state["agent_decision"] == "retry" and state["retry_count"] <= 2:
        return "extract_data"
    else:
        logger.warning("Agent aborted after validation: %s", state["agent_reasoning"])
        return END


# ---------------------------------------------------------------------------
# Build LangGraph Pipeline
# ---------------------------------------------------------------------------
workflow = StateGraph(PipelineState)

workflow.add_node("plan_extraction", plan_extraction)
workflow.add_node("extract_data", extract_data)
workflow.add_node("validate_extraction", validate_extraction)
workflow.add_node("transform_data", transform_data)
workflow.add_node("decide_output", decide_output)

workflow.set_entry_point("plan_extraction")
workflow.add_conditional_edges("plan_extraction", route_after_plan)
workflow.add_edge("extract_data", "validate_extraction")
workflow.add_conditional_edges("validate_extraction", route_after_validation)
workflow.add_edge("transform_data", "decide_output")
workflow.add_edge("decide_output", END)

app = workflow.compile()

# ---------------------------------------------------------------------------
# Entry point
# ---------------------------------------------------------------------------
initial_state: PipelineState = {
    "geojson_path": GEOJSON_PATH,
    "raw_features": [],
    "extraction_plan": {},
    "extracted_data": [],
    "validated_data": [],
    "transformed_data": [],
    "quality_report": {},
    "agent_decision": "",
    "agent_reasoning": "",
    "retry_count": 0,
    "output_path": "",
}

result = app.invoke(initial_state)

print("=== Supermarket Extraction Complete ===")
print(f"Decision:  {result['agent_decision']}")
print(f"Reasoning: {result['agent_reasoning']}")
print(f"Records:   {len(result['transformed_data'])}")
print(f"Output:    {result['output_path']}")

AuthenticationError: Error code: 401 - {'error': {'message': "Incorrect API key provided: 'sk-proj**********************************************************************************************************************************************************Y0A'. You can find your API key at https://platform.openai.com/account/api-keys.", 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}

## 1d. MRT/LRT Station Extraction

Extracts MRT and LRT station locations using a LangGraph pipeline:
- **Data source:** Official station code databases for station names and line information
- **Geocoding:** OneMap API to obtain lat/long coordinates for each station
- **Validation:** LLM agent verifies station count and coordinate bounds
- **Output:** `dataset/transport/mrt_lrt_stations.csv` (~200 rows)

In [ ]:
import os
import json
import requests
import logging
import time
from typing import TypedDict, Literal
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, StructField, StructType, DoubleType

from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

os.environ["SPARK_SUBMIT_OPTS"] = (
    "--add-opens java.base/javax.security.auth=ALL-UNNAMED "
    "--add-opens java.base/sun.nio.ch=ALL-UNNAMED "
    "--add-opens java.base/java.lang=ALL-UNNAMED "
    "--add-opens java.base/java.util=ALL-UNNAMED"
)

spark = (
    SparkSession.builder
    .appName("mrt_lrt_stations_extractor")
    .config("spark.driver.extraJavaOptions",
            "--add-opens java.base/javax.security.auth=ALL-UNNAMED "
            "--add-opens java.base/sun.nio.ch=ALL-UNNAMED "
            "--add-opens java.base/java.lang=ALL-UNNAMED "
            "--add-opens java.base/java.util=ALL-UNNAMED "
            "-Djava.security.manager=allow")
    .config("spark.executor.extraJavaOptions",
            "--add-opens java.base/javax.security.auth=ALL-UNNAMED "
            "-Djava.security.manager=allow")
    .getOrCreate()
)

llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0,
    api_key=load_secret("OPENAI_API_KEY"),
)

class PipelineState(TypedDict):
    station_codes: list[str]
    station_map: list[dict]  # [{"code": "NS1", "name": "Jurong East"}, ...]
    raw_data: list[dict]
    validated_data: list[dict]
    transformed_data: list[dict]
    quality_report: dict
    agent_decision: str
    retry_count: int
    agent_reasoning: str
    output_path: str


def plan_extraction(state: PipelineState) -> PipelineState:
    """Ask the LLM agent to generate all Singapore MRT/LRT station codes and names."""
    all_stations = []

    line_groups = [
        ("North South Line (NS)", "NS"),
        ("North East Line (NE)", "NE"),
        ("East West Line (EW) including Changi Extension (CG)", "EW, CG"),
        ("Circle Line (CC) including Circle Extension (CE)", "CC, CE"),
        ("Thomson-East Coast Line (TE)", "TE"),
        ("Downtown Line (DT)", "DT"),
        ("Bukit Panjang LRT (BP)", "BP"),
        ("Sengkang LRT (STC, SE, SW)", "STC, SE, SW"),
        ("Punggol LRT (PTC, PE, PW)", "PTC, PE, PW"),
    ]

    for line_name, prefixes in line_groups:
        response = llm.invoke([
            SystemMessage(content=(
                "You are a Singapore public transport expert. "
                "List ALL stations for the requested MRT/LRT line. "
                "Respond ONLY with a valid JSON array of objects. "
                'Each object must have "code" and "name" keys. '
                'Example: [{"code": "NS1", "name": "Jurong East"}, {"code": "NS2", "name": "Bukit Batok"}]. '
                "Include all currently operational stations as of 2026. "
                "Do NOT skip any stations."
            )),
            HumanMessage(content=(
                f"List every station code and name for the {line_name}. "
                f"Prefixes used: {prefixes}."
            )),
        ])

        try:
            stations = json.loads(response.content)
            if (isinstance(stations, list) and
                all(isinstance(s, dict) and "code" in s and "name" in s for s in stations)):
                all_stations.extend(stations)
                logger.info("Agent returned %d stations for %s", len(stations), line_name)
            else:
                logger.warning("Agent returned invalid format for %s: %s", line_name, response.content)
        except json.JSONDecodeError:
            logger.warning("Agent returned non-JSON for %s: %s", line_name, response.content)

    # Deduplicate by code
    seen = set()
    unique_stations = []
    for s in all_stations:
        if s["code"] not in seen:
            seen.add(s["code"])
            unique_stations.append(s)

    # Agent validates the combined list
    response = llm.invoke([
        SystemMessage(content=(
            "You are a data engineering agent. Evaluate the station list and "
            "respond ONLY with valid JSON: "
            '{"decision": "proceed"|"abort", "reasoning": "...", "code_count": <int>}'
        )),
        HumanMessage(content=(
            f"I have {len(unique_stations)} stations covering NSL, NEL, EWL, CCL, TEL, DTL, "
            f"BP LRT, Sengkang LRT, Punggol LRT. "
            f"Sample: {unique_stations[:3]}...{unique_stations[-3:]}. "
            f"Should I proceed with extraction?"
        )),
    ])

    try:
        agent_output = json.loads(response.content)
    except json.JSONDecodeError:
        agent_output = {"decision": "proceed", "reasoning": "Defaulting to proceed", "code_count": len(unique_stations)}

    # Override abort if codes are within expected range
    if agent_output["decision"] == "abort" and 150 <= len(unique_stations) <= 300:
        logger.warning("Agent suggested abort but stations look valid (%d). Overriding to proceed.", len(unique_stations))
        agent_output["decision"] = "proceed"
        agent_output["reasoning"] = f"Overridden: {agent_output['reasoning']}"

    logger.info("Agent generated %d total stations", len(unique_stations))
    logger.info("Agent Plan Decision: %s — %s", agent_output["decision"], agent_output["reasoning"])

    state["station_codes"] = [s["code"] for s in unique_stations]
    state["station_map"] = unique_stations
    state["agent_decision"] = agent_output["decision"]
    state["agent_reasoning"] = agent_output["reasoning"]
    return state


def extract_data(state: PipelineState) -> PipelineState:
    """Search OneMap using station name + MRT/LRT suffix for better hit rates."""
    mrt_lrt_data = []
    failed_codes = []

    # Build a code -> name lookup
    code_to_name = {s["code"]: s["name"] for s in state["station_map"]}

    for code in state["station_codes"]:
        station_name = code_to_name.get(code, code)

        # Try multiple search terms: "name MRT STATION", "name LRT STATION", then just code
        search_terms = [
            f"{station_name} MRT STATION",
            f"{station_name} LRT STATION",
            code,
        ]

        found = False
        for search_val in search_terms:
            try:
                url = (
                    f"https://www.onemap.gov.sg/api/common/elastic/search"
                    f"?searchVal={search_val}&returnGeom=Y&getAddrDetails=Y&pageNum=1"
                )
                resp = requests.get(url, timeout=10)

                if resp.status_code != 200:
                    continue

                ctype = resp.headers.get("Content-Type", "")
                if "application/json" not in ctype.lower():
                    continue

                data = resp.json()
                if "results" in data and isinstance(data["results"], list):
                    for result in data["results"]:
                        sv = result.get("SEARCHVAL", "")
                        if "MRT" in sv or "LRT" in sv:
                            result["STATION_CODE"] = code
                            mrt_lrt_data.append(result)
                            found = True
                            break

                if found:
                    break

                time.sleep(0.03)
            except Exception as e:
                logger.warning("Failed to fetch %s (search=%s): %s", code, search_val, e)

        if not found:
            logger.warning("Failed to find station for %s (%s)", code, station_name)
            failed_codes.append(code)

        time.sleep(0.05)

    state["raw_data"] = mrt_lrt_data
    state["quality_report"] = {
        "total_codes": len(state["station_codes"]),
        "extracted": len(mrt_lrt_data),
        "failed": len(failed_codes),
        "failed_codes": failed_codes[:20],
        "success_rate": round(len(mrt_lrt_data) / max(len(state["station_codes"]), 1) * 100, 2),
    }
    logger.info("Extracted %d / %d stations", len(mrt_lrt_data), len(state["station_codes"]))
    return state


def validate_extraction(state: PipelineState) -> PipelineState:
    """Agent reviews extraction quality and decides whether to proceed, retry, or abort."""
    qr = state["quality_report"]

    response = llm.invoke([
        SystemMessage(content=(
            "You are a data quality agent. Evaluate the extraction results and respond "
            "ONLY with valid JSON: "
            '{"decision": "proceed"|"retry"|"abort", "reasoning": "..."}'
            "\nRules: success_rate >= 80% → proceed; 50-80% → retry (max 2); <50% → abort."
        )),
        HumanMessage(content=(
            f"Extraction report: {json.dumps(qr)}. "
            f"Current retry count: {state['retry_count']}. "
            f"Max retries: 2. What should we do?"
        )),
    ])

    try:
        agent_output = json.loads(response.content)
    except json.JSONDecodeError:
        agent_output = {
            "decision": "proceed" if qr["success_rate"] >= 80 else "abort",
            "reasoning": "Fallback rule-based decision",
        }

    state["agent_decision"] = agent_output["decision"]
    state["agent_reasoning"] = agent_output["reasoning"]

    if agent_output["decision"] == "retry":
        state["retry_count"] += 1

    logger.info("Agent QA Decision: %s — %s", agent_output["decision"], agent_output["reasoning"])
    return state


def transform_data(state: PipelineState) -> PipelineState:
    schema = StructType([
        StructField("SEARCHVAL", StringType(), True),
        StructField("BLK_NO", StringType(), True),
        StructField("ROAD_NAME", StringType(), True),
        StructField("BUILDING", StringType(), True),
        StructField("ADDRESS", StringType(), True),
        StructField("POSTAL", StringType(), True),
        StructField("X", StringType(), True),
        StructField("Y", StringType(), True),
        StructField("LATITUDE", StringType(), True),
        StructField("LONGITUDE", StringType(), True),
        StructField("STATION_CODE", StringType(), True),
    ])

    df = spark.createDataFrame(state["raw_data"], schema=schema)
    df = df.dropDuplicates()
    df = df.drop("SEARCHVAL", "BLK_NO", "X", "Y")
    df = (
        df.withColumnRenamed("ROAD_NAME", "roadName")
          .withColumnRenamed("BUILDING", "building")
          .withColumnRenamed("ADDRESS", "address")
          .withColumnRenamed("POSTAL", "postalCode")
          .withColumnRenamed("LATITUDE", "latitude")
          .withColumnRenamed("LONGITUDE", "longitude")
          .withColumnRenamed("STATION_CODE", "stationCode")
    )
    df = df.withColumn("latitude", F.col("latitude").cast(DoubleType()))
    df = df.withColumn("longitude", F.col("longitude").cast(DoubleType()))

    # Derive MRT line from station code
    df = df.withColumn(
        "line",
        F.when(F.col("stationCode").rlike("^NS"), "North South Line")
         .when(F.col("stationCode").rlike("^NE"), "North East Line")
         .when(F.col("stationCode").rlike("^EW|^CG"), "East West Line")
         .when(F.col("stationCode").rlike("^CC|^CE"), "Circle Line")
         .when(F.col("stationCode").rlike("^TE"), "Thomson-East Coast Line")
         .when(F.col("stationCode").rlike("^DT"), "Downtown Line")
         .when(F.col("stationCode").rlike("^BP"), "Bukit Panjang LRT")
         .when(F.col("stationCode").rlike("^SE|^SW|^STC"), "Sengkang LRT")
         .when(F.col("stationCode").rlike("^PE|^PW|^PTC"), "Punggol LRT")
         .otherwise("Unknown")
    )

    state["transformed_data"] = [row.asDict() for row in df.collect()]
    logger.info("Transformed %d station records", df.count())
    return state


def decide_output(state: PipelineState) -> PipelineState:
    """Write cleaned MRT/LRT station data to same directory as this script."""
    output_dir = os.path.join(os.getcwd(), "dataset/transport")
    os.makedirs(output_dir, exist_ok=True)

    path = os.path.join(output_dir, "mrt_lrt_stations.csv")

    df = spark.createDataFrame(state["transformed_data"])
    df.toPandas().to_csv(path, index=False)

    state["output_path"] = path
    state["agent_reasoning"] = "CSV output to dataset/transport."
    logger.info("Output written to %s", path)
    return state


def route_after_plan(state: PipelineState) -> Literal["extract_data", "__end__"]:
    if state["agent_decision"] == "abort":
        logger.warning("Agent aborted pipeline: %s", state["agent_reasoning"])
        return END
    return "extract_data"


def route_after_validation(state: PipelineState) -> Literal["transform_data", "extract_data", "__end__"]:
    if state["agent_decision"] == "proceed":
        return "transform_data"
    elif state["agent_decision"] == "retry" and state["retry_count"] <= 2:
        return "extract_data"
    else:
        logger.warning("Agent aborted after validation: %s", state["agent_reasoning"])
        return END


workflow = StateGraph(PipelineState)

workflow.add_node("plan_extraction", plan_extraction)
workflow.add_node("extract_data", extract_data)
workflow.add_node("validate_extraction", validate_extraction)
workflow.add_node("transform_data", transform_data)
workflow.add_node("decide_output", decide_output)

workflow.set_entry_point("plan_extraction")
workflow.add_conditional_edges("plan_extraction", route_after_plan)
workflow.add_edge("extract_data", "validate_extraction")
workflow.add_conditional_edges("validate_extraction", route_after_validation)
workflow.add_edge("transform_data", "decide_output")
workflow.add_edge("decide_output", END)

app = workflow.compile()

# Section 2: HDB Resale Price Data Loading & Geocoding

Downloads HDB resale price transaction data from the Data.gov.sg API across four time periods (2000-2012, 2012-2014, 2015-2016, 2017-present), aligns schemas across datasets, unions them into a single DataFrame, and geocodes each flat's address using the OneMap API.

> **Note:** The Data.gov.sg API currently provides 4 resale price datasets. The earliest dataset (1990-1999) may need to be added separately if required (5 CSVs, >=1M rows).

## 2a. Data Download Configuration

Defines the Data.gov.sg API configuration, the resale price schema (11 columns: month, town, flat_type, block, street_name, storey_range, floor_area_sqm, flat_model, lease_commence_date, remaining_lease, resale_price), and three helper functions:
- `download_dataset()` - Polls the Data.gov.sg API until data is ready, handles pagination
- `align_to_schema()` - Standardizes column names across dataset versions
- `load_from_api()` - Orchestrates downloading and combining all datasets into one DataFrame

In [ ]:
import time

import requests
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType, IntegerType
)


API_KEY = load_secret("GOV_DATA")

PRICE_DATASETS = [
    {"id": "d_8b84c4ee58e3cfc0ece0d773c8ca6abc", "label": "2017-present"},
    {"id": "d_ea9ed51da2787afaf8e51f827c304208", "label": "2015-2016"},
    {"id": "d_2d5ff9ea31397b66239f245f57751537", "label": "Mar2012-2014"},
    {"id": "d_43f493c6c50d54243cc1eab0df142d6a", "label": "2000-Feb2012"},
]

# fixed schema
PRICE_SCHEMA = StructType([
    StructField("month",               StringType(),  True),  # YYYY-MM
    StructField("town",                StringType(),  True),
    StructField("flat_type",           StringType(),  True),
    StructField("block",               StringType(),  True),
    StructField("street_name",         StringType(),  True),
    StructField("storey_range",        StringType(),  True),  # e.g. "07 TO 09"
    StructField("floor_area_sqm",      DoubleType(),  True),
    StructField("flat_model",          StringType(),  True),
    StructField("lease_commence_date", IntegerType(), True),  # YYYY
    StructField("remaining_lease",     StringType(),  True),  # e.g. "61 years 04 months"
    StructField("resale_price",        DoubleType(),  True),
    StructField("_source",             StringType(),  True),
])

def align_to_schema(df, label):
    """
    Align any dataset to the canonical schema regardless of which
    columns are present. Missing columns are added as null.
    Casts are applied after alignment so types are always consistent.
    """

    # Normalise column names to lowercase + underscores to match schema
    for col in df.columns:
        clean = col.strip().lower().replace(" ", "_")
        if clean != col:
            df = df.withColumnRenamed(col, clean)

    # Add missing columns as null
    canonical_cols = [
        "month", "town", "flat_type", "block", "street_name",
        "storey_range", "floor_area_sqm", "flat_model",
        "lease_commence_date", "remaining_lease", "resale_price"
    ]
    for col in canonical_cols:
        if col not in df.columns:
            print(f"    [{label}] '{col}' not found — adding as null")
            df = df.withColumn(col, F.lit(None).cast(StringType()))

    # Cast numeric columns safely after all columns are present
    df = (
        df
        .withColumn("floor_area_sqm",
                    F.when(F.col("floor_area_sqm").rlike(r"^-?\d+(\.\d+)?$"),
                           F.col("floor_area_sqm").cast(DoubleType()))
                    .otherwise(F.lit(None).cast(DoubleType())))
        .withColumn("lease_commence_date",
                    F.when(F.col("lease_commence_date").rlike(r"^\d+$"),
                           F.col("lease_commence_date").cast(IntegerType()))
                    .otherwise(F.lit(None).cast(IntegerType())))
        .withColumn("resale_price",
                    F.when(F.col("resale_price").rlike(r"^-?\d+(\.\d+)?$"),
                           F.col("resale_price").cast(DoubleType()))
                    .otherwise(F.lit(None).cast(DoubleType())))
    )

    # Keep only canonical columns + source tag in consistent order
    df = df.select(canonical_cols).withColumn("_source", F.lit(label))
    return df


s = requests.Session()
s.headers.update({
    "referer":   "https://colab.research.google.com",
    "x-api-key": API_KEY,
})

def download_dataset(dataset_id, label, schema, max_polls=10):
    print(f"  [{label}]  initiating download...")

    s.get(
        f"https://api-open.data.gov.sg/v1/public/api/datasets/{dataset_id}/initiate-download",
        json={}
    )

    for i in range(max_polls):
        poll = s.get(
            f"https://api-open.data.gov.sg/v1/public/api/datasets/{dataset_id}/poll-download",
            json={}
        ).json()

        if "url" in poll["data"]:
            url = poll["data"]["url"]
            print(f"    ready, downloading...")

            lines = requests.get(url).text.splitlines()
            rdd   = spark.sparkContext.parallelize(lines)
            df    = (spark.read
                         .option("header", "true")
                         .csv(rdd))
            df = df.withColumn("_source", F.lit(label))

            # Align to canonical schema — adds missing columns as null,
            # casts existing columns to correct types
            df = align_to_schema(df, label)
            print(f"    done ✓  ({df.count():,} rows)")
            return df

        print(f"    poll {i+1}/{max_polls}: not ready yet...")
        time.sleep(3)

    raise TimeoutError(f"Dataset {label} never became ready after {max_polls} polls")

def load_from_api(dfs, schema):
    dfs = [download_dataset(ds["id"], ds["label"], schema) for ds in dfs]

    # Union all 4 DataFrames
    from functools import reduce
    combined = reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True), dfs)

    print(f"\n  Total rows loaded: {combined.count():,}")
    return combined

## 2b. Load All Datasets

Calls the API to download all four HDB resale price datasets and unions them into a single `prices_df` DataFrame. This may take a few minutes depending on API response times.

In [ ]:
prices_df = load_from_api(PRICE_DATASETS,PRICE_SCHEMA)

## 2c. Schema Verification

Prints the schema of the combined DataFrame to verify all columns and data types are correct after union.

In [ ]:
prices_df.printSchema()

## 2d. Geocoding HDB Addresses

Authenticates with the OneMap API and defines `geocode_onemap()` to convert a street address to (latitude, longitude) coordinates.

The function calls the OneMap search endpoint with the address, parses the first result's `LATITUDE` and `LONGITUDE` fields, and returns `(None, None)` on failure or no match.

In [ ]:
import requests

url = "https://www.onemap.gov.sg/api/auth/post/getToken"

payload = {
    "email": load_secret("ONEMAP_EMAIL"),
    "password": load_secret("ONEMAP_PASSWORD")
}

response = requests.post(url, json=payload)
data = response.json()

if 'access_token' not in data:
    raise ValueError(f"OneMap auth failed: {data}")

access_token = data['access_token']
access_token

def geocode_onemap(address):
    url = "https://www.onemap.gov.sg/api/common/elastic/search"

    params = {
        "searchVal": address,
        "returnGeom": "Y",
        "getAddrDetails": "Y",
        "pageNum": 1
    }

    try:
        response = requests.get(url, params=params, timeout=5)
        data = response.json()

        if data["found"] > 0:
            result = data["results"][0]
            return float(result["LATITUDE"]), float(result["LONGITUDE"])
        else:
            return None, None
    except:
        return None, None

## 2e. Apply Geocoding (Deduplicated, Cached)

Geocodes HDB flat addresses using a deduplicated, cached approach to avoid the performance and stability issues of a Spark UDF:

1. **Build `full_address`** — concatenates `block` + `street_name` into a single column on `prices_df`
2. **Cache check** — if `dataset/geocoded_addresses.csv` already exists, loads coordinates from disk (skips all API calls on subsequent runs)
3. **Deduplicate** — extracts only distinct `full_address` values via `toPandas()` (~10k unique addresses vs 1M+ rows)
4. **Parallel geocoding** — uses `ThreadPoolExecutor(max_workers=5)` to call `geocode_onemap()` concurrently; logs progress every 100 addresses
5. **Persist cache** — saves results to `dataset/geocoded_addresses.csv` so this step only runs once
6. **Join back** — converts the geocoded pandas DataFrame to a Spark DataFrame and left-joins on `full_address`

> **Why not a Spark UDF?** UDFs run inside Spark worker processes, which crash under sustained HTTP load (`Python worker exited unexpectedly`). Running geocoding outside Spark (pure Python threads) is more stable and far more efficient due to address deduplication.

In [ ]:
import sys
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from pyspark.sql.functions import concat_ws

GEOCACHE_PATH = "dataset/geocoded_addresses.csv"

# Build full_address column
prices_df = prices_df.withColumn("full_address", concat_ws(" ", prices_df.block, prices_df.street_name))

if os.path.exists(GEOCACHE_PATH):
    print("Loading geocoded addresses from cache..."); sys.stdout.flush()
    geo_pd = pd.read_csv(GEOCACHE_PATH)
    print(f"Loaded {len(geo_pd):,} cached addresses"); sys.stdout.flush()
else:
    print("Geocoding unique addresses (first run only)..."); sys.stdout.flush()
    addr_pd = prices_df.select("full_address").toPandas()
    unique_addresses = addr_pd["full_address"].dropna().unique().tolist()
    print(f"Unique addresses to geocode: {len(unique_addresses):,}"); sys.stdout.flush()

    results = {}

    def geocode_one(address):
        lat, lng = geocode_onemap(address)
        return address, lat, lng

    with ThreadPoolExecutor(max_workers=5) as executor:
        futures = {executor.submit(geocode_one, addr): addr for addr in unique_addresses}
        for i, future in enumerate(as_completed(futures)):
            address, lat, lng = future.result()
            results[address] = (lat, lng)
            if (i + 1) % 100 == 0:
                print(f"  Geocoded {i + 1:,}/{len(unique_addresses):,}"); sys.stdout.flush()

    geo_pd = pd.DataFrame([
        {"full_address": addr, "latitude": lat, "longitude": lng}
        for addr, (lat, lng) in results.items()
    ])

    os.makedirs("dataset", exist_ok=True)
    geo_pd.to_csv(GEOCACHE_PATH, index=False)
    print(f"Saved {len(geo_pd):,} geocoded addresses to {GEOCACHE_PATH}"); sys.stdout.flush()

# Join geocoded coordinates back to prices_df
geo_spark = spark.createDataFrame(geo_pd)
prices_df = prices_df.join(geo_spark, on="full_address", how="left")

prices_df = prices_df.cache()
print(f"Geocoded DataFrame cached: {prices_df.count():,} rows")
prices_df.show(10)

# Section 3: LLM-Guided Data Cleaning Agent

Implements an automated data cleaning pipeline powered by an OpenAI LLM (gpt-4o-mini). The pipeline follows four steps:
1. **Profile** the DataFrame to understand column types, null percentages, and value distributions
2. **Plan** cleaning steps by asking the LLM to select from a registry of 14 cleaning tools
3. **Execute** the plan with automatic error recovery (the LLM diagnoses failures and suggests fixes, up to 2 retries per tool)
4. **Post-process** with deterministic validation rules, outlier removal (IQR method), and deduplication

## 3a. Dependencies and Initialization

Installs LangGraph, LangChain, and visualization libraries (matplotlib, seaborn). Then imports the OpenAI client and PySpark type definitions needed for the cleaning tools.

In [ ]:
!pip install -q langgraph langchain langchain-openai matplotlib seaborn

In [ ]:
import json
import openai
from pyspark.sql.types import (
    StringType, LongType,
    FloatType, DateType, TimestampType
)
NUMERIC_TYPES = (DoubleType, FloatType, IntegerType, LongType)

In [ ]:
client = openai.OpenAI(api_key=load_secret("OPENAI_API_KEY"))
MAX_RETRIES = 2

## 3b. Cleaning Tool Registry

Defines 14 reusable PySpark-based cleaning functions, each operating on a DataFrame column:
- **String normalization:** `trim_uppercase` - trims whitespace and uppercases string columns
- **Date parsing:** `parse_yyyymm_date`, `parse_yyyymmdd_date` - convert date strings to DateType
- **Numeric extraction:** `parse_storey_range` ("04 TO 06" -> 5.0), `parse_lease_string` ("61 years 04 months" -> 61.33)
- **Derived features:** `derive_year_month` (extracts transaction_year, transaction_month via substring — avoids implicit `to_timestamp` in Spark 3.4+), `compute_remaining_lease` (99 - age)
- **Null handling:** `drop_nulls`, `impute_remaining_lease`
- **Outlier detection:** `flag_outliers_zscore`, `flag_outliers_iqr` (IQR preferred per project spec)
- **Standardization:** `standardise_values` (maps inconsistent strings to canonical forms)
- **Column management:** `drop_column`

> **Spark 3.4+ note:** `_transaction_month_to_date()` uses `F.lit(None).cast(DateType())` in the `otherwise` branch instead of `try_to_date(s)`. In Spark 3.4+, columnar evaluation evaluates all `when/otherwise` branches for the entire column before applying the condition mask — `try_to_date(s)` without a format string internally calls `to_timestamp`, which throws `DateTimeException` on `YYYY-MM` strings.

In [ ]:
def _transaction_month_to_date(column_name):
    """
    Parse a transaction-month column whether values are YYYY-MM, YYYY-MM-DD,
    or already Date/Timestamp. Avoids appending '-01' to a full date (which
    yields invalid strings like '2017-01-01-01').
    """
    s = F.trim(F.col(column_name).cast("string"))
    return (
        F.when(s.rlike(r"^\d{4}-\d{2}-\d{2}"), F.to_date(s, "yyyy-MM-dd"))
        .when(s.rlike(r"^\d{4}-\d{2}$"), F.to_date(F.concat(s, F.lit("-01")), "yyyy-MM-dd"))
        .otherwise(F.lit(None).cast(DateType()))
    )


def tool_trim_uppercase(sdf, column, **kwargs):
    """Trim whitespace and upper-case a string column."""
    return sdf.withColumn(column, F.upper(F.trim(F.col(column))))

def tool_parse_yyyymm_date(sdf, column, **kwargs):
    """
    Parse a YYYY-MM (or YYYY-MM-DD) column into DateType (first of month).
    Replaces the original column in-place.
    """
    return sdf.withColumn(column, _transaction_month_to_date(column))

def tool_parse_yyyymmdd_date(sdf, column, **kwargs):
    """Parse a YYYY-MM-DD string column into a DateType column."""
    return sdf.withColumn(column, F.to_date(F.col(column), "yyyy-MM-dd"))

def tool_parse_lease_string(sdf, column, **kwargs):
    """
    Parse remaining_lease string → numeric years, replacing the original column.
    Handles: '61 years 04 months', '62 years 01 month', '63 years'
    """
    years = F.nullif(
        F.regexp_extract(F.col(column), r"(\d+)\s*[Yy]ear", 1), F.lit("")
    ).cast(DoubleType())

    months = F.coalesce(
        F.nullif(F.regexp_extract(F.col(column), r"(\d+)\s*[Mm]onth", 1), F.lit(""))
         .cast(DoubleType()),
        F.lit(0.0)
    )
    # Replace original column
    return sdf.withColumn(column, F.coalesce(years, F.lit(None).cast(DoubleType())) + months / 12.0)

def tool_drop_nulls(sdf, column, **kwargs):
    """Drop rows where the specified column is null or empty string."""
    return sdf.filter(
        F.col(column).isNotNull() & (F.trim(F.col(column).cast("string")) != "")
    )

def tool_flag_outliers_zscore(sdf, column, group_by=None, threshold=4.0, **kwargs):
    """
    Add a boolean column {column}_outlier flagging rows where the z-score
    of `column` exceeds `threshold` (default 4.0).
    If group_by is provided, z-score is computed within each group.
    Rows are flagged, NOT removed.
    """
    out_col = column + "_outlier"
    w = Window.partitionBy(group_by) if group_by else Window.rowsBetween(
        Window.unboundedPreceding, Window.unboundedFollowing
    )
    return (
        sdf
        .withColumn("_mean", F.mean(column).over(w))
        .withColumn("_std",  F.stddev(column).over(w))
        .withColumn(
            out_col,
            F.when(
                F.col("_std") > 0,
                F.abs(F.col(column) - F.col("_mean")) / F.col("_std") > threshold
            ).otherwise(F.lit(False))
        )
        .drop("_mean", "_std")
    )

def tool_standardise_values(sdf, column, replacements, **kwargs):
    """
    Replace inconsistent string values in a column.
    `replacements` is a dict: {"old_value": "new_value", ...}
    e.g. {"MULTI GENERATION": "MULTI-GENERATION", "MULTI-GEN": "MULTI-GENERATION"}
    """
    for old, new in replacements.items():
        sdf = sdf.withColumn(
            column,
            F.when(F.col(column) == old, F.lit(new)).otherwise(F.col(column))
        )
    return sdf

def tool_impute_remaining_lease(sdf, column, **kwargs):
    """
    Impute missing remaining_lease values using:
        99 - (transaction_year - lease_commence_date)
    Only fills nulls — existing values are preserved.
    Requires: lease_commence_date (int) and month (YYYY-MM string) to be present.
    """
    sale_year = F.year(_transaction_month_to_date("month"))
    approximated = (99 - (sale_year - F.col("lease_commence_date"))).cast(DoubleType())

    return sdf.withColumn(
        column,
        F.coalesce(F.col(column), approximated)   # keep existing, fill nulls
    )
def tool_parse_storey_range(sdf, column, **kwargs):
    """
    Parse 'XX TO YY' storey_range string into a numeric storey_midpoint column.
    e.g. "04 TO 06" → storey_midpoint = 5.0
    """
    lo = F.regexp_extract(F.col(column), r"^(\d+)", 1).cast("double")
    hi = F.regexp_extract(F.col(column), r"(\d+)$", 1).cast("double")
    return sdf.withColumn("storey_midpoint", (lo + hi) / 2)


def tool_derive_year_month(sdf, column, **kwargs):
    """
    Extract transaction_year (int) and transaction_month (int) from a YYYY-MM column.
    Uses substring extraction instead of F.year()/F.month() to avoid implicit
    to_timestamp coercion in Spark 3.4+ which fails on short strings like '2017-01'.
    Works for both StringType ('YYYY-MM') and DateType (cast to 'YYYY-MM-DD').
    """
    s = F.trim(F.col(column).cast("string"))
    return (sdf
        .withColumn("transaction_year",  F.substring(s, 1, 4).cast("int"))
        .withColumn("transaction_month", F.substring(s, 6, 2).cast("int"))
    )


def tool_flag_outliers_iqr(sdf, column, multiplier=1.5, **kwargs):
    """
    Flag outliers using the IQR method. Adds a boolean column {column}_outlier.
    lower_bound = Q1 - multiplier * IQR
    upper_bound = Q3 + multiplier * IQR
    Rows are flagged, NOT removed here (removal happens in post-processing).
    """
    quantiles = sdf.approxQuantile(column, [0.25, 0.75], 0.01)
    q1, q3 = quantiles[0], quantiles[1]
    iqr = q3 - q1
    lower = q1 - multiplier * iqr
    upper = q3 + multiplier * iqr
    out_col = f"{column}_outlier"
    return sdf.withColumn(
        out_col,
        (F.col(column) < lower) | (F.col(column) > upper)
    )


def tool_compute_remaining_lease(sdf, column, **kwargs):
    """
    Always compute remaining_lease_years from lease_commence_date.
    Formula: 99 - (transaction_year - lease_commence_date)
    Overwrites the column for ALL rows — no string parsing needed.
    Reliable for all 4 datasets including pre-2015 data where remaining_lease is null.
    Requires: transaction_year (int) and lease_commence_date (int) columns.
    Run AFTER derive_year_month.
    """
    computed = (99 - (F.col("transaction_year") - F.col("lease_commence_date"))).cast(DoubleType())
    return sdf.withColumn("remaining_lease_years", computed)


def tool_drop_column(sdf, column, **kwargs):
    """Drop a column nominated by the agent in columns_to_drop."""
    return sdf.drop(column)


In [ ]:
# Registry — maps tool name (string) → function
TOOL_REGISTRY = {
    "trim_uppercase":         tool_trim_uppercase,
    "parse_yyyymm_date":      tool_parse_yyyymm_date,
    "parse_yyyymmdd_date":    tool_parse_yyyymmdd_date,
    "parse_lease_string":     tool_parse_lease_string,
    "drop_nulls":             tool_drop_nulls,
    "flag_outliers_zscore":   tool_flag_outliers_zscore,
    "flag_outliers_iqr":      tool_flag_outliers_iqr,
    "standardise_values":     tool_standardise_values,
    "impute_remaining_lease": tool_impute_remaining_lease,
    "parse_storey_range":     tool_parse_storey_range,
    "derive_year_month":      tool_derive_year_month,
    "compute_remaining_lease": tool_compute_remaining_lease,
    "drop_column":             tool_drop_column,
}

TOOL_CATALOGUE = """
Available cleaning tools (name → what it does → required args → optional args):
  trim_uppercase          → trim whitespace and upper-case a string column
                            args: column
  parse_yyyymm_date       → parse YYYY-MM string → DateType (first of month)
                            args: column
  parse_yyyymmdd_date     → parse YYYY-MM-DD string → DateType
                            args: column
  parse_lease_string      → extract years from '61 years 04 months' or plain '61'
                            → replaces column with double (years)
                            args: column
  drop_nulls              → drop rows where column is null or empty
                            args: column
  flag_outliers_iqr       → PREFERRED outlier method: add {col}_outlier boolean using IQR
                            lower = Q1 - multiplier*IQR, upper = Q3 + multiplier*IQR
                            args: column   optional: multiplier (float, default 1.5)
                            USE THIS for resale_price (required by project spec)
  flag_outliers_zscore    → add {col}_outlier boolean (z-score > threshold, default 4.0)
                            args: column   optional: group_by (column name), threshold (float)
  standardise_values      → replace inconsistent string values
                            args: column, replacements (dict of old→new)
  impute_remaining_lease  → fill null remaining_lease values using
                            99 - (sale_year - lease_commence_date)
                            args: column
                            requires: month and lease_commence_date columns
  parse_storey_range      → parse 'XX TO YY' string → numeric storey_midpoint column
                            args: column (must be storey_range)
  derive_year_month       → extract transaction_year (int) and transaction_month (int)
                            from a YYYY-MM column
                            args: column (must be month)
  compute_remaining_lease → always compute remaining_lease_years = 99 - (transaction_year - lease_commence_date)
                            args: column (ignored — always writes remaining_lease_years)
                            requires: transaction_year and lease_commence_date columns
                            run AFTER derive_year_month
  drop_column            → drop a column nominated by the agent (columns_to_drop list)
                            args: column
"""

## 3c. DataFrame Profiler

`profile_dataframe()` generates a text summary of the DataFrame for the LLM. It includes:
- Total row count
- Per-column: type, null count (%), distinct count, min/max for numerics, sample values
- Pattern detection (e.g., YYYY-MM format, "X TO Y" ranges, unit strings)

This profile is sent to the LLM so it can make informed decisions about which cleaning tools to apply.

In [ ]:
def profile_dataframe(sdf):
    """Profile a Spark DataFrame in ~3 actions (not per-column).

    Batches null counts, distinct counts, numeric min/max, and string
    pattern detection into single aggregation passes to avoid the
    50-70 full-scan problem of the per-column approach.
    """
    # ── 1. Count (DataFrame should already be cached upstream) ──
    total = sdf.count()
    sample_rows = sdf.limit(3).collect()

    # ── 2. Batch aggregation: nulls + numeric min/max + string patterns (1 action) ──
    agg_exprs = []
    for field in sdf.schema.fields:
        col = field.name
        # Null count
        if isinstance(field.dataType, StringType):
            agg_exprs.append(
                F.sum((F.col(col).isNull() | (F.trim(F.col(col)) == "")).cast("int")).alias(f"{col}__nulls")
            )
        else:
            agg_exprs.append(
                F.sum(F.col(col).isNull().cast("int")).alias(f"{col}__nulls")
            )
        # Numeric min/max
        if isinstance(field.dataType, NUMERIC_TYPES):
            agg_exprs.append(F.min(col).alias(f"{col}__min"))
            agg_exprs.append(F.max(col).alias(f"{col}__max"))
        # String pattern detection
        if isinstance(field.dataType, StringType):
            non_null_expr = F.col(col).isNotNull() & (F.trim(F.col(col)) != "")
            agg_exprs.append(
                F.sum(non_null_expr.cast("int")).alias(f"{col}__non_null")
            )
            agg_exprs.append(
                F.sum((non_null_expr & F.col(col).rlike(r"^\d{4}-\d{2}$")).cast("int")).alias(f"{col}__pat_yyyymm")
            )
            agg_exprs.append(
                F.sum((non_null_expr & F.col(col).rlike(r"^\d+ TO \d+$")).cast("int")).alias(f"{col}__pat_xtoy")
            )
            agg_exprs.append(
                F.sum((non_null_expr & F.col(col).rlike(r"\d+\s*(year|month)")).cast("int")).alias(f"{col}__pat_unit")
            )

    stats_row = sdf.agg(*agg_exprs).collect()[0]

    # ── 3. Approximate distinct counts (1 action, uses HyperLogLog) ──
    approx_exprs = [
        F.approx_count_distinct(F.col(field.name)).alias(field.name)
        for field in sdf.schema.fields
    ]
    distinct_row = sdf.agg(*approx_exprs).collect()[0]

    # ── Build output ──
    lines = [
        f"Total rows: {total:,}",
        "",
        f"{'Column':<25} {'Type':<15} {'Nulls':>8} {'Distinct':>10}  {'Patterns':<30}  {'Samples'}",
        "-" * 110
    ]

    for field in sdf.schema.fields:
        col = field.name
        dtype = str(field.dataType)

        null_n = stats_row[f"{col}__nulls"]
        null_str = f"{null_n/total*100:.0f}%" if null_n > 0 else "0%"

        distinct_n = distinct_row[col]

        patterns = ""
        if isinstance(field.dataType, StringType):
            nn = stats_row[f"{col}__non_null"]
            if nn and nn > 0:
                checks = [
                    ("YYYY-MM",  stats_row[f"{col}__pat_yyyymm"] / nn * 100),
                    ("X TO Y",   stats_row[f"{col}__pat_xtoy"]   / nn * 100),
                    ("unit str", stats_row[f"{col}__pat_unit"]    / nn * 100),
                ]
                patterns = "  ".join(f"{label}:{v:.0f}%" for label, v in checks if v > 10)

        if isinstance(field.dataType, NUMERIC_TYPES):
            patterns = f"min={stats_row[f'{col}__min']}  max={stats_row[f'{col}__max']}"

        samples = ", ".join(str(r[col]) for r in sample_rows if r[col] is not None)[:50]

        lines.append(
            f"{col:<25} {dtype:<15} {null_str:>8} {distinct_n:>10}  {patterns:<30}  {samples}"
        )

    return "\n".join(lines)

## 3d. LLM Configuration and Error Recovery

Three functions that interact with the OpenAI API:
- `ask_llm_for_config(profile_str)` - Sends the DataFrame profile to gpt-4o-mini and receives a structured JSON cleaning plan (outlier method, threshold, null handling strategy, columns to drop)
- `config_to_plan(cleaning_config)` - Converts the JSON config into an ordered list of (tool_name, column, kwargs) tuples for sequential execution
- `ask_llm_to_fix(error_type, error_msg, failed_call, profile_str)` - When a tool fails during execution, asks the LLM to diagnose the error and suggest a revised tool call

In [ ]:
def ask_llm_for_config(profile_str):
    """
    Ask the LLM to return a structured cleaning_config.
    The LLM decides the strategy; config_to_plan() translates it to tool calls,
    ensuring PySpark cleaning is driven by the agent's config — not hardcoded.
    """
    prompt = f"""You are a data cleaning expert for an HDB resale price ML pipeline.
Analyse the DataFrame profile and return a cleaning configuration JSON.

{TOOL_CATALOGUE}

DATAFRAME PROFILE:
{profile_str}

Return ONLY valid JSON with exactly these six keys:

{{
  "storey_range_strategy": "parse_storey_range",
  "outlier_method": "IQR",
  "outlier_threshold": <float, choose 1.5 to 3.0 based on price spread>,
  "null_handling": {{
    "<column_name>": "drop_nulls" | "impute_remaining_lease"
  }},
  "columns_to_drop": ["<col>"],
  "duplicate_strategy": "dropDuplicates"
}}

Rules:
  - storey_range_strategy: always "parse_storey_range"
  - outlier_method: always "IQR" (required by project spec)
  - outlier_threshold: between 1.5 and 3.0 — choose based on the resale_price spread
  - null_handling: only include columns that have >0 nulls in the profile;
    use "impute_remaining_lease" for remaining_lease, "drop_nulls" for others
  - columns_to_drop: list columns that add no predictive value (e.g. _source)
  - duplicate_strategy: always "dropDuplicates"
"""
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.1,
    )
    raw = resp.choices[0].message.content.strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
    return json.loads(raw)


def config_to_plan(cleaning_config):
    """
    Convert a cleaning_config dict into an ordered list of
    tool calls compatible with TOOL_REGISTRY.
    This is how PySpark cleaning is driven by the agent's config.
    """
    plan = []

    # 1. Trim + uppercase all string categoricals
    for col in ["town", "flat_type", "flat_model", "street_name", "block"]:
        plan.append({"tool": "trim_uppercase", "column": col, "kwargs": {},
                     "reason": "Normalise categorical text"})

    # 2. Parse month -> DateType, then extract integer year/month
    plan.append({"tool": "parse_yyyymm_date", "column": "month", "kwargs": {},
                 "reason": "Convert YYYY-MM string to DateType"})
    plan.append({"tool": "derive_year_month", "column": "month", "kwargs": {},
                 "reason": "Extract transaction_year and transaction_month"})

    # 3. Storey range -> midpoint (driven by config)
    if cleaning_config.get("storey_range_strategy") == "parse_storey_range":
        plan.append({"tool": "parse_storey_range", "column": "storey_range", "kwargs": {},
                     "reason": "Parse storey_range string to numeric storey_midpoint"})

    # 4. Flat type standardisation (canonical values)
    plan.append({
        "tool": "standardise_values", "column": "flat_type",
        "kwargs": {"replacements": {
            "MULTI GENERATION": "MULTI-GENERATION",
            "MULTI-GEN": "MULTI-GENERATION",
            "multi generation": "MULTI-GENERATION",
        }},
        "reason": "Standardise flat_type to canonical values"
    })

    # 5. Compute remaining_lease_years from lease_commence_date (reliable for all 4 datasets)
    # Replaces parse_lease_string + impute_remaining_lease — handles null columns for pre-2015 data
    plan.append({"tool": "compute_remaining_lease", "column": "remaining_lease", "kwargs": {},
                 "reason": "Compute remaining_lease_years = 99 - (transaction_year - lease_commence_date) for all rows"})
    # Drop original remaining_lease string column — always superseded by remaining_lease_years
    plan.append({"tool": "drop_column", "column": "remaining_lease", "kwargs": {},
                 "reason": "Drop original remaining_lease string column (replaced by remaining_lease_years)"})

    # 6. Null handling per config (agent decides which columns need drop_nulls)
    for col, strategy in cleaning_config.get("null_handling", {}).items():
        if strategy == "drop_nulls" and col != "remaining_lease":
            plan.append({"tool": "drop_nulls", "column": col, "kwargs": {},
                         "reason": f"Remove null rows in {col} (agent decision from cleaning_config)"})

    # 6b. Drop columns nominated by agent (columns_to_drop)
    for col in cleaning_config.get("columns_to_drop", []):
        plan.append({"tool": "drop_column", "column": col, "kwargs": {},
                     "reason": f"Drop {col} per agent config — no predictive value"})

    # 7. Outlier flagging driven by config method + threshold
    method    = cleaning_config.get("outlier_method", "IQR")
    threshold = float(cleaning_config.get("outlier_threshold", 1.5))
    if method == "IQR":
        plan.append({"tool": "flag_outliers_iqr", "column": "resale_price",
                     "kwargs": {"multiplier": threshold},
                     "reason": f"Flag resale_price outliers using IQR (multiplier={threshold}) per agent config"})
    else:
        plan.append({"tool": "flag_outliers_zscore", "column": "resale_price",
                     "kwargs": {"threshold": threshold},
                     "reason": f"Flag resale_price outliers using z-score (threshold={threshold}) per agent config"})

    return plan


def ask_llm_to_fix(error_type, error_msg, failed_call, profile_str):
    prompt = f"""A cleaning tool call has failed. Revise the arguments to fix it.

FAILED CALL:
{json.dumps(failed_call, indent=2)}

ERROR TYPE   : {error_type}
ERROR MESSAGE: {error_msg}

{TOOL_CATALOGUE}

DATAFRAME PROFILE:
{profile_str}

Rules:
  - Revise only the arguments of the failed call — do not change the intent
  - If the column type is wrong for the tool, suggest the correct tool instead
  - Do not suggest null handling tools — those are handled separately by the human
  - Do not suggest impute_remaining_lease unless the original call was already
    related to remaining_lease

Return ONLY valid JSON, no markdown:
{{
  "diagnosis": "one sentence root cause",
  "revised_call": {{
    "tool": "tool_name",
    "column": "column_name",
    "kwargs": {{}}
  }}
}}"""

    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
    )
    raw = resp.choices[0].message.content.strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
    return json.loads(raw)


## 3e. Pipeline Execution

Runs the full cleaning pipeline:
1. Profiles `prices_df` and displays null counts before cleaning
2. Calls `ask_llm_for_config()` to get the LLM's cleaning strategy
3. Saves the cleaning decisions to `cleaning_decisions.md`
4. Converts config to a tool execution plan via `config_to_plan()`
5. Executes each tool sequentially; on failure, calls `ask_llm_to_fix()` for up to 2 retries
6. Tracks the current DataFrame state as `current_sdf`

In [ ]:
print("\nStep 1: Profiling DataFrame...")
profile_str = profile_dataframe(prices_df)
print(profile_str)

# Null counts BEFORE cleaning
print("\n=== Null counts BEFORE cleaning ===")
prices_df.select(
    [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in prices_df.columns]
).show(vertical=True)

print("\nStep 2: Asking LLM for cleaning configuration...")
cleaning_config = ask_llm_for_config(profile_str)
print("\nCleaning config (agent decision):")
print(json.dumps(cleaning_config, indent=2))

# Save cleaning_decisions.md immediately after LLM returns config
import datetime
with open("cleaning_decisions.md", "w") as f:
    f.write("# Cleaning Decisions\n\n")
    f.write(f"**Agent:** Data Cleaning Agent (gpt-4o-mini)\n")
    f.write(f"**Timestamp:** {datetime.datetime.now().isoformat()}\n\n")
    f.write("## Cleaning Config\n\n")
    f.write(f"```json\n{json.dumps(cleaning_config, indent=2)}\n```\n")
print("\ncleaning_decisions.md written ✓")

# Convert config to ordered tool-call plan (PySpark reads config via this function)
plan = config_to_plan(cleaning_config)
print(f"\nDerived plan — {len(plan)} tool calls:")
for i, call in enumerate(plan, 1):
    kwargs_str = f"  kwargs={call['kwargs']}" if call.get("kwargs") else ""
    print(f"  {i}. {call['tool']}({call['column']}){kwargs_str}")
    print(f"     → {call['reason']}")

fix_log     = []
current_sdf = prices_df

print("\nStep 3: Executing plan...")
print("=" * 52)

for call in plan:
    tool_name    = call["tool"]
    column       = call["column"]
    kwargs       = call.get("kwargs", {})
    current_call = call.copy()

    print(f"\n  {tool_name}({column})")

    if tool_name not in TOOL_REGISTRY:
        print(f"  SKIPPED - unknown tool '{tool_name}'")
        continue

    success = False

    for attempt in range(1, MAX_RETRIES + 2):
        try:
            result_sdf = TOOL_REGISTRY[tool_name](current_sdf, column, **kwargs)
            result_sdf.count()   # force Spark evaluation

            print(f"  Attempt {attempt}: PASSED")
            current_sdf = result_sdf
            success = True
            break

        except Exception as e:
            err_type = type(e).__name__
            err_msg  = str(e)[:300]
            print(f"  Attempt {attempt}: FAILED  ({err_type}: {err_msg[:80]})")

            if attempt > MAX_RETRIES:
                print("  Max retries reached. Skipping.")
                break

            print("  Asking LLM to revise call...")
            llm_result   = ask_llm_to_fix(err_type, err_msg, current_call, profile_str)
            diagnosis    = llm_result.get("diagnosis", "Unknown")
            revised      = llm_result.get("revised_call", {})
            tool_name    = revised.get("tool", tool_name)
            column       = revised.get("column", column)
            kwargs       = revised.get("kwargs", kwargs)
            current_call = revised

            print(f"  Diagnosis    : {diagnosis}")
            print(f"  Revised call : {tool_name}({column}) kwargs={kwargs}")

            fix_log.append({
                "original": call,
                "attempt":  attempt,
                "error":    err_msg[:150],
                "diagnosis": diagnosis,
                "revised":  revised,
            })

    if not success:
        print(f"  Could not complete after {MAX_RETRIES} retries.")

## 3f. Post-Processing and Validation

After the LLM-guided pipeline completes, applies deterministic validation rules:
- **Whitelist check:** `flat_type` must be one of {1 ROOM, 2 ROOM, 3 ROOM, 4 ROOM, 5 ROOM, EXECUTIVE, MULTI-GENERATION}
- **Range checks:** `floor_area_sqm > 0`, `resale_price > 0`, `remaining_lease_years >= 0`, `transaction_year >= 2000`
- **Outlier removal:** Drops rows flagged by the IQR method
- **Deduplication:** Removes exact duplicate rows

The final cleaned DataFrame is assigned to `clean_df` for downstream use in EDA and ML.

In [ ]:
print("PIPELINE FINISHED")
print(f"Total automatic revisions: {len(fix_log)}")

print("\nFINAL SCHEMA")
current_sdf.printSchema()

print("\nFINAL SAMPLE")
current_sdf.show(5, truncate=False)

if fix_log:
    print("\nFIX LOG")
    print("-" * 52)
    for i, fix in enumerate(fix_log, 1):
        orig = fix["original"]
        print(f"\nRevision {i} — {orig['tool']}({orig['column']})  attempt {fix['attempt']}")
        print(f"  Error    : {fix['error']}")
        print(f"  Diagnosis: {fix['diagnosis']}")
        print(f"  Revised  : {fix['revised']}")

clean_df = current_sdf
print(f"\nclean_df ready: {clean_df.count():,} rows, {len(clean_df.columns)} columns")

# ── POST-PROCESSING (deterministic) ─────────────────────────────────────────

# 0. Deterministic flat_type whitelist validation
CANONICAL_FLAT_TYPES = {"1 ROOM", "2 ROOM", "3 ROOM", "4 ROOM", "5 ROOM", "EXECUTIVE", "MULTI-GENERATION"}
n_invalid = clean_df.filter(~F.col("flat_type").isin(list(CANONICAL_FLAT_TYPES))).count()
if n_invalid > 0:
    print(f"\nWarning: {n_invalid:,} rows with non-canonical flat_type — filtering out")
    clean_df = clean_df.filter(F.col("flat_type").isin(list(CANONICAL_FLAT_TYPES)))
else:
    print("flat_type: all values are canonical ✓")

# 1. Validate floor_area_sqm — no negative values
n_neg = clean_df.filter(F.col("floor_area_sqm") <= 0).count()
if n_neg > 0:
    print(f"Removing {n_neg:,} rows with non-positive floor_area_sqm")
    clean_df = clean_df.filter(F.col("floor_area_sqm") > 0)

# 2. Remove exact duplicates
n_before_dedup = clean_df.count()
clean_df = clean_df.dropDuplicates()
n_after_dedup = clean_df.count()
print(f"\nDuplicates removed: {n_before_dedup - n_after_dedup:,} rows ({n_before_dedup:,} → {n_after_dedup:,})")

# 3. Remove outliers flagged by IQR
outlier_col = "resale_price_outlier"
n_removed = 0
if outlier_col in clean_df.columns:
    n_before_out = clean_df.count()
    clean_df = clean_df.filter(~F.col(outlier_col)).drop(outlier_col)
    n_after_out = clean_df.count()
    n_removed = n_before_out - n_after_out
    print(f"Outliers removed  : {n_removed:,} rows ({n_removed/n_before_out*100:.1f}%) — IQR method on resale_price")

# 4. Null counts AFTER cleaning
print("\n=== Null counts AFTER cleaning ===")
clean_df.select(
    [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in clean_df.columns]
).show(vertical=True)

# 5. Consolidated cleaning summary
n_raw = prices_df.count()  # actual raw row count
n_final = clean_df.count()
print("\n=== Cleaning Summary (Before vs After) ===")
print(f"{'Stage':<35} {'Rows':>10}  {'Change':>20}")
print("-" * 68)
print(f"{'Raw rows loaded':<35} {n_raw:>10,}  {'—':>20}")
print(f"{'After deduplication':<35} {n_after_dedup:>10,}  {f'-{n_before_dedup - n_after_dedup:,} dupes':>20}")
print(f"{'After outlier removal':<35} {n_final:>10,}  {f'-{n_removed:,} outliers':>20}")
print(f"{'Total rows removed':<35} {n_raw - n_final:>10,}  {f'{(n_raw - n_final)/n_raw*100:.1f}% of raw':>20}")
print(f"\nclean_df final: {n_final:,} rows, {len(clean_df.columns)} columns")


# Section 4: Exploratory Data Analysis

Three visualizations on the cleaned dataset to understand price patterns before feature engineering:
1. **Price trends over time** — Median resale price by flat type with 6-month rolling average
2. **Price distributions** — Violin plots by flat type; box plots for top-10 towns by median price
3. **Feature correlation heatmap** — Pearson correlations between numeric features

## 4a. Price Trends Over Time

Median resale price by flat type with a 6-month rolling average line chart to identify temporal trends across the 2000–2025 transaction window.

In [ ]:
# ── EDA 1: Price trends over time ──
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

df = clean_df

trend_pdf = (
    df
      .groupBy("month", "flat_type")
      .agg(F.percentile_approx("resale_price", 0.5).alias("median_price"))
      .toPandas()
)

trend_pdf["month"] = pd.to_datetime(trend_pdf["month"])
trend_pdf = trend_pdf.sort_values("month")

trend_pdf["median_price"] = (
    trend_pdf.groupby("flat_type")["median_price"]
             .transform(lambda x: x.rolling(6, min_periods=1).mean())
)

fig, ax = plt.subplots(figsize=(13,5))

for ft, grp in trend_pdf.groupby("flat_type"):
    ax.plot(grp["month"], grp["median_price"]/1e3, linewidth=2, label=ft)

import matplotlib.dates as mdates
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

ax.set_title("Median HDB Resale Price Trend by Flat Type", fontsize=14, fontweight="bold")
ax.set_ylabel("Median Price (SGD '000)")
ax.set_xlabel("Year")

ax.legend(title="Flat Type", bbox_to_anchor=(1.02,1))
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 4b. Price Distribution by Flat Type and Town

Violin plots showing price distribution across flat types and box plots for the top-10 towns ranked by median resale price.

In [ ]:
# ── EDA 2: Price distribution by flat type & town ──
dist_pdf = (
    df
      .select("resale_price", "flat_type", "town")
      .toPandas()
)

# Violin by flat_type
fig, ax = plt.subplots(figsize=(12, 5))
order = dist_pdf.groupby("flat_type")["resale_price"].median().sort_values().index
sns.violinplot(data=dist_pdf, x="flat_type", y="resale_price",
               order=order, palette="muted", ax=ax, inner="quartile")
ax.set_title("Resale Price Distribution by Flat Type", fontsize=13, fontweight="bold")
ax.set_xlabel("Flat Type")
ax.set_ylabel("Resale Price (SGD)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x/1e3:.0f}k"))
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

# Box by top-10 towns
top10 = dist_pdf.groupby("town")["resale_price"].median().nlargest(10).index
fig, ax = plt.subplots(figsize=(13, 5))
sns.boxplot(data=dist_pdf[dist_pdf["town"].isin(top10)], x="town", y="resale_price",
            order=top10, palette="Set2", ax=ax, flierprops={"markersize": 2})
ax.set_title("Resale Price Distribution — Top 10 Towns by Median Price",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Town")
ax.set_ylabel("Resale Price (SGD)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x/1e3:.0f}k"))
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

## 4c. Feature Correlation Heatmap

Pearson correlation matrix for numeric features (`resale_price`, `floor_area_sqm`, `lease_commence_date`, etc.) to identify collinear pairs before modeling.

In [ ]:
# ── EDA 3: Correlation heatmap ──
corr_cols = [c for c in [
    "resale_price", "floor_area_sqm", "lease_commence_date",
    "storey_midpoint", "transaction_year", "transaction_month", "remaining_lease_years"
] if c in df.columns]

corr_pdf = df.select(corr_cols).dropna().limit(50_000).toPandas()
corr_pdf = corr_pdf.select_dtypes(include="number")
corr_mat = corr_pdf.corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr_mat, dtype=bool))
sns.heatmap(corr_mat, mask=mask, annot=True, fmt=".2f",
            cmap="RdYlGn", center=0, vmin=-1, vmax=1,
            linewidths=0.5, ax=ax, annot_kws={"size": 9})
ax.set_title("Feature Correlation Heatmap", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print("\n  Correlations with resale_price (absolute, descending):")
print(
    corr_mat["resale_price"]
    .drop("resale_price")
    .abs()
    .sort_values(ascending=False)
    .to_string()
)

# Section 5: Feature Engineering & Preprocessing

Transforms `clean_df` into a model-ready DataFrame through four stages:
- **5a.** Time-based features (year, quarter, remaining lease)
- **5b.** Categorical encoding via StringIndexer + OneHotEncoder pipeline
- **5c.** Interaction features specified by the Feature Engineering Agent contract
- **5d.** Multicollinearity analysis (correlation matrix, VIF, feature selection)

## 5a. Time-Based Feature Engineering

Create and validate temporal features for trend modeling:
- `transaction_year` (integer)
- `transaction_quarter` (1-4)
- `years_since_lease_start = transaction_year - lease_commence_date`
- `remaining_lease_years` sanity check against `99 - years_since_lease_start` for standard HDB leases

These features are derived only from time/calendar and lease metadata (no target leakage).

In [ ]:
# Build/standardize time-based features on clean_df
from pyspark.sql import functions as F

# Robust year extraction from month whether it is string (YYYY-MM) or date-like.
transaction_year_expr = F.coalesce(
    F.col("transaction_year").cast("int") if "transaction_year" in clean_df.columns else F.lit(None).cast("int"),
    F.year(F.to_date(F.col("month"))),
    F.substring(F.col("month").cast("string"), 1, 4).cast("int")
)

df_time = (
    clean_df
    .withColumn("transaction_year", transaction_year_expr.cast("int"))
)

# If transaction_month exists, reuse it. Otherwise derive from month.
if "transaction_month" in df_time.columns:
    month_expr = F.col("transaction_month").cast("int")
else:
    month_expr = F.coalesce(
        F.month(F.to_date(F.col("month"))),
        F.substring(F.col("month").cast("string"), 6, 2).cast("int")
    )

df_time = (
    df_time
    .withColumn("transaction_month", month_expr.cast("int"))
    .withColumn("transaction_quarter", (((F.col("transaction_month") - 1) / 3).cast("int") + 1).cast("int"))
    .withColumn("years_since_lease_start", (F.col("transaction_year") - F.col("lease_commence_date")).cast("double"))
)

# Ensure remaining_lease_years exists and aligns with standard 99-year lease logic.
if "remaining_lease_years" in df_time.columns:
    df_time = df_time.withColumn(
        "remaining_lease_years",
        F.coalesce(
            F.col("remaining_lease_years").cast("double"),
            (F.lit(99.0) - F.col("years_since_lease_start")).cast("double")
        )
    )
else:
    df_time = df_time.withColumn(
        "remaining_lease_years",
        (F.lit(99.0) - F.col("years_since_lease_start")).cast("double")
    )

# Final non-null numeric guarantees for AC
# (fallback to 0 only when source metadata is missing/unparseable)
df_time = df_time.na.fill({
    "transaction_year": 0,
    "transaction_month": 1,
    "transaction_quarter": 1,
    "years_since_lease_start": 0.0,
    "remaining_lease_years": 0.0,
})

clean_df = df_time
print("Added/updated time features on clean_df:")
print([c for c in [
    "transaction_year", "transaction_quarter", "years_since_lease_start", "remaining_lease_years"
] if c in clean_df.columns])

In [ ]:
# Validation checks for acceptance criteria
from pyspark.sql import functions as F

required_time_cols = [
    "transaction_year",
    "transaction_quarter",
    "years_since_lease_start",
    "remaining_lease_years",
]

# 1) Null checks
null_counts = clean_df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c) for c in required_time_cols
]).toPandas().iloc[0].to_dict()

# 2) Numeric checks
numeric_types = {c: dict(clean_df.dtypes).get(c) for c in required_time_cols}

# 3) Quarter range checks
quarter_out_of_range = clean_df.filter((F.col("transaction_quarter") < 1) | (F.col("transaction_quarter") > 4)).count()

# 4) Standard HDB sanity check: remaining_lease_years ~ 99 - years_since_lease_start
sanity_df = clean_df.withColumn(
    "lease_formula_gap",
    F.abs(F.col("remaining_lease_years") - (F.lit(99.0) - F.col("years_since_lease_start")))
)
gap_stats = sanity_df.select(
    F.avg("lease_formula_gap").alias("avg_gap"),
    F.expr("percentile_approx(lease_formula_gap, 0.5)").alias("median_gap"),
    F.max("lease_formula_gap").alias("max_gap")
).toPandas().iloc[0].to_dict()

print("=== Time Feature Validation ===")
print("Numeric dtypes:", numeric_types)
print("Null counts:", null_counts)
print("Quarter out-of-range rows:", quarter_out_of_range)
print("Lease sanity gap stats (years):", gap_stats)
print("Leakage check: PASS (features use only month/lease fields, not resale_price)")

clean_df.select(required_time_cols).show(5, truncate=False)

## 5b. Categorical Encoding for Spark MLlib

Encode `town`, `flat_type`, `flat_model`, and `storey_range_bin` using `StringIndexer` + `OneHotEncoder` in a reproducible PySpark `Pipeline`.

Design choices:
- Fit on **training data only** to avoid label leakage
- Apply fitted pipeline model to test data
- Keep explicit `*_index` and `*_ohe` columns for traceability
- Document distinct category counts and index-to-label mapping

In [ ]:
# Train/test split + categorical encoding pipeline (fit on train only)
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder

base_df = clean_df

# Ensure required categorical columns exist. If storey_range_bin is absent,
# derive a coarse bin from storey_midpoint for compatibility.
if "storey_range_bin" not in base_df.columns:
    if "storey_midpoint" in base_df.columns:
        base_df = base_df.withColumn(
            "storey_range_bin",
            F.when(F.col("storey_midpoint") <= 6, F.lit("low"))
             .when(F.col("storey_midpoint") <= 12, F.lit("mid"))
             .otherwise(F.lit("high"))
        )
    else:
        base_df = base_df.withColumn("storey_range_bin", F.lit("unknown"))

categorical_cols = ["town", "flat_type", "flat_model", "storey_range_bin"]

# Ensure string type and no nulls for robust indexing.
for c in categorical_cols:
    base_df = base_df.withColumn(c, F.coalesce(F.col(c).cast("string"), F.lit("unknown")))

train_df, test_df = base_df.randomSplit([0.8, 0.2], seed=42)

index_output_cols = [f"{c}_index" for c in categorical_cols]
ohe_output_cols = [f"{c}_ohe" for c in categorical_cols]

indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_index", handleInvalid="keep")
    for c in categorical_cols
]
encoder = OneHotEncoder(
    inputCols=index_output_cols,
    outputCols=ohe_output_cols,
    handleInvalid="keep",
    dropLast=True,
)

cat_pipeline = Pipeline(stages=[*indexers, encoder])
cat_pipeline_model = cat_pipeline.fit(train_df)  # no leakage: fit only on train

train_encoded_df = cat_pipeline_model.transform(train_df)
test_encoded_df = cat_pipeline_model.transform(test_df)

print("Categorical encoding pipeline fitted on training data only.")
print("Index columns:", index_output_cols)
print("OHE columns:", ohe_output_cols)
print("Train rows:", train_encoded_df.count(), "| Test rows:", test_encoded_df.count())

In [ ]:
# Documentation: distinct category counts + traceable index-to-label mapping
from pyspark.ml.feature import StringIndexerModel

# Distinct categories documented from training split (the fitting population).
category_counts = {
    c: train_df.select(c).distinct().count()
    for c in categorical_cols
}

print("=== Distinct Category Counts (train split) ===")
for c, n in category_counts.items():
    print(f"{c}: {n}")

# Build traceable mapping from index to original category labels.
indexer_models = [s for s in cat_pipeline_model.stages if isinstance(s, StringIndexerModel)]
label_mappings = {}
for m in indexer_models:
    original_col = m.getInputCol()
    labels = list(m.labels)
    label_mappings[original_col] = {idx: label for idx, label in enumerate(labels)}

print("\n=== Index-to-Category Mapping ===")
for col_name, mapping in label_mappings.items():
    print(f"\n{col_name} ({len(mapping)} labels)")
    # Show first 15 labels for readability; full mapping remains available in label_mappings.
    for idx, label in list(mapping.items())[:15]:
        print(f"  {idx} -> {label}")

print("\nLeakage check: PASS (StringIndexer/OneHotEncoder fitted on train_df only, then applied to test_df)")

# Optional quick schema preview of encoded outputs
train_encoded_df.select(
    "town", "town_index", "town_ohe",
    "flat_type", "flat_type_index", "flat_type_ohe"
).show(5, truncate=False)

## 5c. Interaction Features

The **Feature Engineering Agent** emits structured `feature_config` with an `interaction_features` array. PySpark code below uses that JSON to create numeric interaction columns and attach them to the **MLlib feature vector** (`features`).

- `floor_area_x_remaining_lease` = `floor_area_sqm` × `remaining_lease_years` (depreciation × size)
- `town_x_flat_type_idx` = `town_index` × `flat_type_index` (town × flat type after `StringIndexer`)

The same schema is defined in **Section 6** `feature_engineering_agent` so the LangGraph node and this section stay aligned.

In [ ]:
import json
from pyspark.sql import functions as F

# Default structured output of feature_engineering_agent (Section 4) — keep definitions in sync.
FEATURE_CONFIG_DEFAULT = {
    "distance_features": ["mrt", "school", "supermarket", "hawker"],
    "school_quality_features": ["ip", "sap", "autonomous"],
    "economic_features": ["hdb_rpi", "price_to_rpi_ratio"],
    "demographic_features": [
        "median_household_income",
        "total_population",
        "price_to_income_ratio",
        "income_yoy_growth",
        "population_yoy_growth",
    ],
    "time_features": [],
    "categorical_features": [],
    "binning": {},
    "interaction_features": [
        {
            "name": "floor_area_x_remaining_lease",
            "kind": "multiply",
            "operands": ["floor_area_sqm", "remaining_lease_years"],
            "description": "Floor area \u00d7 remaining lease; proxies depreciation-adjusted space value",
        },
        {
            "name": "town_x_flat_type_idx",
            "kind": "indexed_product",
            "operands": ["town_index", "flat_type_index"],
            "description": "Town \u00d7 flat_type via StringIndexer indices (numeric surrogate for categorical cross)",
        },
    ],
    "features_to_drop": [],
}

print("Agent feature_config JSON:\n")
print(json.dumps({k: v for k, v in FEATURE_CONFIG_DEFAULT.items() if k != "interaction_features"}, indent=2))
print("\ninteraction_features:")
print(json.dumps(FEATURE_CONFIG_DEFAULT["interaction_features"], indent=2))


def apply_interactions_from_config(df, cfg):
    """Create interaction columns from feature_config.interaction_features (numeric only)."""
    out = df
    created = []
    for spec in cfg.get("interaction_features", []):
        kind = spec.get("kind")
        name = spec.get("name")
        ops = spec.get("operands", [])
        if not name or len(ops) != 2:
            continue
        a, b = ops[0], ops[1]
        if kind == "multiply" and a in out.columns and b in out.columns:
            out = out.withColumn(name, F.col(a).cast("double") * F.col(b).cast("double"))
            created.append(name)
        elif kind == "indexed_product" and a in out.columns and b in out.columns:
            out = out.withColumn(name, F.col(a).cast("double") * F.col(b).cast("double"))
            created.append(name)
    return out, created


train_encoded_ix, interaction_cols_created = apply_interactions_from_config(
    train_encoded_df, FEATURE_CONFIG_DEFAULT
)
test_encoded_ix, _ = apply_interactions_from_config(test_encoded_df, FEATURE_CONFIG_DEFAULT)

assert len(interaction_cols_created) >= 2, "Expected at least two interaction features from agent config"
assert "floor_area_x_remaining_lease" in train_encoded_ix.columns

print("Interactions created:", interaction_cols_created)
print("Next cell: correlation / VIF \u2192 drops \u2192 VectorAssembler(features)")

## 5d. Multicollinearity Analysis

- Correlation matrix for **numeric** predictors via `Correlation.corr` on a `VectorAssembler` column (PySpark ML).
- Heatmap uses **pandas/matplotlib only for plotting** (`.toPandas()` on the correlation matrix or a small sample is acceptable).
- Pairs with **|r| > 0.85** are listed; we **drop** redundant predictors with a short justification (e.g. lease-age definitions that are algebraically tied).
- Final modeling feature set is documented in a table.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation

CORR_THRESHOLD = 0.85

# Numeric predictors only (exclude target, IDs, and sparse vector columns)
num_cols = [
    c
    for c, t in train_encoded_ix.dtypes
    if t in ("int", "bigint", "double", "float", "smallint")
    and c not in ("resale_price", "flat_id")
    and not c.endswith("_ohe")
]

num_cols = [c for c in num_cols if c in train_encoded_ix.columns]

corr_assembler = VectorAssembler(inputCols=num_cols, outputCol="_corr_vec", handleInvalid="skip")
corr_ready = corr_assembler.transform(train_encoded_ix.na.drop(subset=num_cols))
corr_mat = Correlation.corr(corr_ready, "_corr_vec").head()[0]
corr_arr = corr_mat.toArray()

corr_pdf = pd.DataFrame(corr_arr, index=num_cols, columns=num_cols)

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(
    corr_pdf,
    mask=np.triu(np.ones_like(corr_pdf, dtype=bool), k=1),
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    linewidths=0.3,
    ax=ax,
)
ax.set_title("Numeric feature correlation matrix (train, PySpark Correlation.corr)")
plt.tight_layout()
plt.show()

high_corr_pairs = []
for i in range(len(num_cols)):
    for j in range(i + 1, len(num_cols)):
        r = float(corr_arr[i, j])
        if abs(r) > CORR_THRESHOLD:
            high_corr_pairs.append((num_cols[i], num_cols[j], r))

print(f"\nPairs with |correlation| > {CORR_THRESHOLD}:")
for a, b, r in sorted(high_corr_pairs, key=lambda x: -abs(x[2])):
    print(f"  {a} vs {b}: {r:.4f}")

# --- Justified drops (multicollinearity) ---
# years_since_lease_start and remaining_lease_years are both derived from lease timeline;
# for standard 99-year HDB they are nearly linear transforms of each other → keep one.
multicollinearity_drops = []
for a, b, r in high_corr_pairs:
    pair = {a, b}
    if pair == {"years_since_lease_start", "remaining_lease_years"}:
        multicollinearity_drops.append("years_since_lease_start")
    if pair == {"lease_commence_date", "remaining_lease_years"}:
        multicollinearity_drops.append("lease_commence_date")

multicollinearity_drops = sorted(set(multicollinearity_drops))
if not multicollinearity_drops and high_corr_pairs:
    # Fallback: document first extreme pair (manual review)
    print(
        "\nNote: No automatic drop rule matched. Review high_corr_pairs above before modeling."
    )

print("\nColumns dropped for multicollinearity:", multicollinearity_drops)
print(
    "Justification: retain remaining_lease_years as the direct policy-relevant lease measure; "
    "years_since_lease_start is redundant once remaining lease is included (and aligns with 99-year structure)."
)

train_model_df = train_encoded_ix.drop(*multicollinearity_drops) if multicollinearity_drops else train_encoded_ix
test_model_df = test_encoded_ix.drop(*multicollinearity_drops) if multicollinearity_drops else test_encoded_ix

# --- VIF (pandas sample; statsmodels) ---
try:
    from statsmodels.stats.outliers_influence import variance_inflation_factor
except ImportError:
    import subprocess
    import sys

    subprocess.check_call([sys.executable, "-m", "pip", "install", "statsmodels", "-q"])
    from statsmodels.stats.outliers_influence import variance_inflation_factor

vif_cols = [c for c in num_cols if c not in multicollinearity_drops]
vif_sample = train_model_df.select(*vif_cols).sample(False, 0.03, seed=42).limit(50_000).toPandas()
vif_sample = vif_sample.dropna()
vif_sample = vif_sample.loc[:, vif_sample.nunique() > 1]

if vif_sample.shape[1] < 2 or vif_sample.shape[0] < 10:
    print("\nVIF skipped: insufficient columns or rows after filtering.")
    vif_pdf = pd.DataFrame()
else:
    try:
        vif_pdf = pd.DataFrame(
            {
                "feature": list(vif_sample.columns),
                "VIF": [
                    variance_inflation_factor(vif_sample.values, i)
                    for i in range(vif_sample.shape[1])
                ],
            }
        ).sort_values("VIF", ascending=False)
        print("\nVIF (sampled train rows; VIF > 10 often flagged as high):")
        print(vif_pdf.to_string(index=False))
    except Exception as exc:
        vif_pdf = pd.DataFrame()
        print("\nVIF skipped (singular or ill-conditioned matrix):", exc)

# --- Final feature vector after drops ---
_vector_inputs = [
    "floor_area_sqm",
    "remaining_lease_years",
    "years_since_lease_start",
    "transaction_year",
    "transaction_quarter",
    "storey_midpoint",
    "floor_area_x_remaining_lease",
    "town_x_flat_type_idx",
    "town_ohe",
    "flat_type_ohe",
    "flat_model_ohe",
    "storey_range_bin_ohe",
]
vector_input_cols = [c for c in _vector_inputs if c in train_model_df.columns]

final_assembler = VectorAssembler(
    inputCols=vector_input_cols,
    outputCol="features",
    handleInvalid="skip",
)
train_feat_df = final_assembler.transform(train_model_df)
test_feat_df = final_assembler.transform(test_model_df)

# --- Final feature documentation table ---
DESCRIPTIONS = {
    "floor_area_sqm": "Floor area (sqm)",
    "remaining_lease_years": "Approx. years left on 99-year lease",
    "years_since_lease_start": "Years since lease commencement",
    "transaction_year": "Sale year",
    "transaction_quarter": "Calendar quarter of sale",
    "storey_midpoint": "Midpoint of storey range",
    "floor_area_x_remaining_lease": "Interaction: size × remaining lease",
    "town_x_flat_type_idx": "Interaction: town index × flat type index",
    "town_ohe": "One-hot town",
    "flat_type_ohe": "One-hot flat type",
    "flat_model_ohe": "One-hot flat model",
    "storey_range_bin_ohe": "One-hot storey bin",
}

doc_rows = []
for c in vector_input_cols:
    doc_rows.append(
        {
            "feature": c,
            "type": dict(train_model_df.dtypes).get(c, "vector"),
            "description": DESCRIPTIONS.get(c, "Model input"),
        }
    )

feature_doc_df = pd.DataFrame(doc_rows)
print("\n=== Final modeling feature set ===")
print(feature_doc_df.to_string(index=False))

# Section 6: LangGraph ML Pipeline

Defines a LangGraph StateGraph pipeline that orchestrates the ML workflow. The pipeline accepts `clean_df` from Section 3 and proceeds through 4 nodes:

1. **`feature_engineering_agent`** - LLM recommends feature transformations (distance to amenities, school quality, economic indicators, demographics, interactions) based on the cleaned data summary
2. **`apply_feature_engineering`** - PySpark creates the recommended features:
   - **Distance features:** Haversine distance to nearest MRT, school, supermarket, hawker centre
   - **School quality features:** Distance to nearest IP/SAP/autonomous school + binary within-2km indicators
   - **Economic features:** HDB Resale Price Index joined by quarter + price-to-RPI ratio
   - **Demographic features:** Median household income, total population, price-to-income ratio, YoY growth rates
   - **Interaction features:** Configurable multiply/indexed-product terms from feature config
3. **`train_models`** - Trains 3 regression models using PySpark MLlib: LinearRegression, RandomForestRegressor, GBTRegressor. Computes RMSE, MAE, R-squared, and MAPE
4. **`evaluation_agent`** - LLM produces a comparative evaluation narrative identifying the best model with reasoning

**AgentState** flows between nodes carrying: `dataset_info`, `feature_config`, `feature_summary`, `model_results`, `evaluation_report`, and `agent_logs`.

> **Status:** Node functions contain stub implementations (TODO placeholders) for model training and evaluation agent skills.

## 6a. Imports, LLM Client, and AgentState

Imports LangGraph, LangChain, and PySpark dependencies. Initialises the OpenAI LLM client (`gpt-4o`, temperature 0). Defines `AgentState` (the TypedDict flowing through the graph) and a `_log_entry` helper for audit logging.

In [ ]:
import logging
from datetime import datetime
from typing import Any, TypedDict

from langgraph.graph import StateGraph, END, START
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from pyspark.sql import functions as F
from pyspark.sql.functions import broadcast
from pyspark.sql.window import Window

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0,
    api_key=load_secret("OPENAI_API_KEY"),
)


# ---------------------------------------------------------------------------
# AgentState
# ---------------------------------------------------------------------------

class AgentState(TypedDict):
    """Shared state flowing through the LangGraph pipeline.

    Cleaning is handled by Section 3 (LLM-guided cleaning agent).
    This pipeline starts from the already-cleaned DataFrame (clean_df).
    """

    # Input: dataset metadata populated before pipeline starts
    dataset_info: dict          # schema, row count, column types, null counts, sample values

    # Feature engineering stage
    feature_config: dict        # JSON from FE agent (distance_features, time_features, etc.)
    feature_summary: dict       # summary of features created
    engineered_df: Any          # DataFrame after feature engineering

    # Model training stage
    model_results: dict         # RMSE, MAE, R-squared, MAPE per model

    # Evaluation stage
    evaluation_report: dict     # best_model, reasoning, key_findings, recommendations

    # Logging
    agent_logs: list            # append-only log of agent decisions and timestamps


# ---------------------------------------------------------------------------
# Helper
# ---------------------------------------------------------------------------

def _log_entry(agent_name: str, detail: str) -> dict:
    return {
        "agent": agent_name,
        "timestamp": datetime.now().isoformat(),
        "detail": detail,
    }



## 6b. Node 1 — Feature Engineering Agent

LLM agent that inspects the cleaned data summary and returns a structured `feature_config` JSON specifying which distance, school-quality, economic, demographic, and interaction features to create.

In [ ]:
# ---------------------------------------------------------------------------
# Node 1: feature_engineering_agent
# ---------------------------------------------------------------------------

def feature_engineering_agent(state: AgentState) -> dict:
    """LLM agent that recommends feature engineering from cleaned data summary."""
    logger.info("Node: feature_engineering_agent")

    # TODO: implement full agent skill
    # Expected: call OpenAI with dataset_info (cleaned data summary), return JSON with keys:
    #   distance_features, time_features, categorical_features,
    #   binning, interaction_features, features_to_drop

    # Default distance feature set aligned with agent output contract.
    # interaction_features: structured list consumed by apply_feature_engineering (and post-encoding cells).
    feature_config = {
        "distance_features": ["mrt", "school", "supermarket", "hawker"],
        "school_quality_features": ["ip", "sap", "autonomous"],
        "economic_features": ["hdb_rpi", "price_to_rpi_ratio"],
        "demographic_features": [
            "median_household_income",
            "total_population",
            "price_to_income_ratio",
            "income_yoy_growth",
            "population_yoy_growth",
        ],
        "time_features": [],
        "categorical_features": [],
        "binning": {},
        "interaction_features": [
            {
                "name": "floor_area_x_remaining_lease",
                "kind": "multiply",
                "operands": ["floor_area_sqm", "remaining_lease_years"],
                "description": "Floor area \u00d7 remaining lease; proxies depreciation-adjusted space value",
            },
            {
                "name": "town_x_flat_type_idx",
                "kind": "indexed_product",
                "operands": ["town_index", "flat_type_index"],
                "description": "Town \u00d7 flat_type via StringIndexer columns (apply after encoding pipeline)",
            },
        ],
        "features_to_drop": [],
    }

    logs = list(state.get("agent_logs", []))
    logs.append(
        _log_entry(
            "feature_engineering_agent",
            "Structured feature_config JSON with distance_features + school_quality + economic + demographic + interaction_features",
        )
    )

    return {"feature_config": feature_config, "agent_logs": logs}



## 6c. Node 2 — Apply Feature Engineering

PySpark node that executes the `feature_config` from Node 1:
- **Distance features** — Haversine nearest-distance to MRT, school, supermarket, hawker centre
- **School quality features** — Nearest IP/SAP/autonomous school distance + binary within-2 km indicators
- **Economic features** — HDB Resale Price Index joined by quarter; price-to-RPI ratio
- **Demographic features** — Median household income, total population, YoY growth rates, price-to-income ratio
- **Interaction features** — Configurable multiply/indexed-product terms

All joins use `broadcast()` for small lookup tables to avoid expensive shuffles.

In [ ]:
# ---------------------------------------------------------------------------
# Node 2: apply_feature_engineering
# ---------------------------------------------------------------------------

def apply_feature_engineering(state: AgentState) -> dict:
    """PySpark node that creates features based on feature_config."""
    logger.info("Node: apply_feature_engineering")

    config = state["feature_config"]

    def _resolve_lat_lon(df, label: str):
        lat_candidates = ["latitude", "lat", f"{label}_lat"]
        lon_candidates = ["longitude", "lng", "lon", f"{label}_lng", f"{label}_lon"]

        lat_col = next((c for c in lat_candidates if c in df.columns), None)
        lon_col = next((c for c in lon_candidates if c in df.columns), None)
        if lat_col is None or lon_col is None:
            raise ValueError(f"Cannot find lat/lon columns for {label}. Columns: {df.columns}")
        return lat_col, lon_col

    def _haversine_km(lat1, lon1, lat2, lon2):
        # Earth radius in km.
        r = F.lit(6371.0)
        lat1_rad = F.radians(lat1)
        lon1_rad = F.radians(lon1)
        lat2_rad = F.radians(lat2)
        lon2_rad = F.radians(lon2)

        dlat = lat2_rad - lat1_rad
        dlon = lon2_rad - lon1_rad
        a = F.pow(F.sin(dlat / 2), 2) + F.cos(lat1_rad) * F.cos(lat2_rad) * F.pow(F.sin(dlon / 2), 2)
        c = 2 * F.atan2(F.sqrt(a), F.sqrt(1 - a))
        return r * c

    def _nearest_distance(df_flats, df_amenity, amenity_label: str, output_col: str):
        flat_lat_col, flat_lon_col = _resolve_lat_lon(df_flats, "flat")
        amen_lat_col, amen_lon_col = _resolve_lat_lon(df_amenity, amenity_label)

        flats = (
            df_flats
            .select("flat_id", F.col(flat_lat_col).cast("double").alias("flat_lat"), F.col(flat_lon_col).cast("double").alias("flat_lon"))
            .filter(F.col("flat_lat").isNotNull() & F.col("flat_lon").isNotNull())
        )
        amenity = (
            df_amenity
            .select(F.col(amen_lat_col).cast("double").alias("amen_lat"), F.col(amen_lon_col).cast("double").alias("amen_lon"))
            .filter(F.col("amen_lat").isNotNull() & F.col("amen_lon").isNotNull())
            .dropDuplicates()
        )

        if amenity.rdd.isEmpty():
            raise ValueError(f"Amenity table '{amenity_label}' is empty after lat/lon cleaning.")

        dist_df = (
            flats.crossJoin(broadcast(amenity))
            .withColumn(output_col, _haversine_km(F.col("flat_lat"), F.col("flat_lon"), F.col("amen_lat"), F.col("amen_lon")))
            .groupBy("flat_id")
            .agg(F.min(F.col(output_col)).cast("double").alias(output_col))
        )

        return df_flats.join(dist_df, on="flat_id", how="left")

    base_df = clean_df

    # Ensure there is a stable row-level key for joins.
    if "flat_id" not in base_df.columns:
        base_df = base_df.withColumn("flat_id", F.monotonically_increasing_id())

    distance_features = set(config.get("distance_features", []))
    if not distance_features:
        distance_features = {"mrt", "school", "supermarket", "hawker"}

    amenity_sources = {
        "mrt": {
            "path": "dataset/transport/mrt_lrt_stations.csv",
            "label": "mrt",
            "output": "dist_nearest_mrt",
        },
        "school": {
            "path": "dataset/school/schools.csv",
            "label": "school",
            "output": "dist_nearest_school",
        },
        "supermarket": {
            "path": "dataset/supermarket/supermarkets.csv",
            "label": "supermarket",
            "output": "dist_nearest_supermarket",
        },
        "hawker": {
            "path": "dataset/hawker_centre/hawker_centres.csv",
            "label": "hawker",
            "output": "dist_nearest_hawker",
        },
    }

    df = base_df
    created_features = []

    for feat in ["mrt", "school", "supermarket", "hawker"]:
        if feat not in distance_features:
            continue

        source = amenity_sources[feat]
        amenity_df = spark.read.option("header", True).csv(source["path"])
        df = _nearest_distance(df, amenity_df, source["label"], source["output"])
        df = df.withColumn(source["output"], F.coalesce(F.col(source["output"]).cast("double"), F.lit(0.0)))
        created_features.append(source["output"])

    # Defensive checks for AC: numeric, non-negative, non-null.
    for col_name in created_features:
        df = df.withColumn(col_name, F.when(F.col(col_name) < 0, F.lit(0.0)).otherwise(F.col(col_name)).cast("double"))

    # -----------------------------------------------------------------------
    # School Quality Features (IP, SAP, Autonomous/Affiliated)
    # -----------------------------------------------------------------------
    school_quality_features = set(config.get("school_quality_features", []))
    if school_quality_features:
        logger.info("Engineering school quality features: %s", school_quality_features)

        schools_df = spark.read.option("header", True).csv("dataset/school/schools.csv")

        school_type_map = {
            "ip":         {"filter_col": "ip_ind",         "dist_col": "dist_nearest_ip_school",   "binary_col": "has_ip_school_nearby"},
            "sap":        {"filter_col": "sap_ind",        "dist_col": "dist_nearest_sap_school",  "binary_col": "has_sap_school_nearby"},
            "autonomous": {"filter_col": "autonomous_ind", "dist_col": "dist_nearest_auto_school", "binary_col": "has_affiliated_school_nearby"},
        }

        for stype in ["ip", "sap", "autonomous"]:
            if stype not in school_quality_features:
                continue
            info = school_type_map[stype]
            filtered_schools = schools_df.filter(F.col(info["filter_col"]) == "Yes")

            if filtered_schools.rdd.isEmpty():
                logger.warning("No %s schools found — skipping %s features", stype, stype)
                continue

            df = _nearest_distance(df, filtered_schools, "school", info["dist_col"])
            df = df.withColumn(info["dist_col"], F.coalesce(F.col(info["dist_col"]).cast("double"), F.lit(0.0)))
            df = df.withColumn(info["dist_col"], F.when(F.col(info["dist_col"]) < 0, F.lit(0.0)).otherwise(F.col(info["dist_col"])).cast("double"))

            # Binary: 1 if nearest school of this type is within 2 km
            df = df.withColumn(info["binary_col"], F.when(F.col(info["dist_col"]) <= 2.0, F.lit(1)).otherwise(F.lit(0)).cast("int"))

            created_features.extend([info["dist_col"], info["binary_col"]])

        logger.info("School quality features created: %s", [c for c in created_features if "school" in c.lower() or "affiliated" in c.lower()])

    # -----------------------------------------------------------------------
    # HDB Resale Price Index (Economic Indicator)
    # -----------------------------------------------------------------------
    economic_features = config.get("economic_features", [])
    if economic_features:
        logger.info("Engineering economic features (HDB RPI)")

        # Data source: HDB Resale Price Index (Quarterly), SingStat Table Builder
        # https://tablebuilder.singstat.gov.sg/table/TS/M212161
        rpi_df = spark.read.option("header", True).csv(
            "dataset/resale_price_index/HDBResalePriceIndex1Q2009100Quarterly.csv"
        )
        # Parse quarter column: format "YYYY-QN" → year (int), quarter (int)
        rpi_df = (
            rpi_df
            .withColumn("rpi_year", F.substring("quarter", 1, 4).cast("int"))
            .withColumn("rpi_quarter", F.substring("quarter", 7, 1).cast("int"))
            .withColumn("rpi_value", F.col("index").cast("double"))
            .select("rpi_year", "rpi_quarter", "rpi_value")
            .filter(F.col("rpi_value").isNotNull())
        )

        # Join on transaction_year and transaction_quarter
        df = df.join(
            broadcast(rpi_df),
            (F.col("transaction_year") == F.col("rpi_year")) & (F.col("transaction_quarter") == F.col("rpi_quarter")),
            "left",
        )
        df = df.withColumn("hdb_rpi", F.coalesce(F.col("rpi_value"), F.lit(0.0)).cast("double"))
        df = df.drop("rpi_year", "rpi_quarter", "rpi_value")

        # Derived: price_to_rpi_ratio = resale_price / hdb_rpi
        df = df.withColumn(
            "price_to_rpi_ratio",
            F.when(F.col("hdb_rpi") > 0, F.col("resale_price").cast("double") / F.col("hdb_rpi"))
             .otherwise(F.lit(0.0))
             .cast("double"),
        )

        created_features.extend(["hdb_rpi", "price_to_rpi_ratio"])
        logger.info("HDB RPI features created: hdb_rpi, price_to_rpi_ratio")

    # -----------------------------------------------------------------------
    # Population Demographics (Income, Population)
    # -----------------------------------------------------------------------
    demographic_features = config.get("demographic_features", [])
    if demographic_features:
        logger.info("Engineering demographic features (income, population)")

        # Data sources:
        # Median Gross Monthly Income from Employment: https://tablebuilder.singstat.gov.sg/table/TS/M810361
        # Total Resident Population: https://tablebuilder.singstat.gov.sg/table/TS/M810001
        demo_df = spark.read.option("header", True).csv("dataset/demographics/demographics_features.csv")
        demo_df = (
            demo_df
            .withColumn("demo_year", F.col("year").cast("int"))
            .withColumn("median_household_income", F.col("median_household_income").cast("double"))
            .withColumn("total_population", F.col("total_population").cast("double"))
        )

        # YoY growth rates (computed on small demographics table before join)
        w = Window.orderBy("demo_year")
        demo_df = demo_df.withColumn(
            "income_yoy_growth",
            F.when(
                F.lag("median_household_income").over(w).isNotNull(),
                (F.col("median_household_income") - F.lag("median_household_income").over(w))
                / F.lag("median_household_income").over(w),
            ).otherwise(F.lit(0.0)).cast("double"),
        )
        demo_df = demo_df.withColumn(
            "population_yoy_growth",
            F.when(
                F.lag("total_population").over(w).isNotNull(),
                (F.col("total_population") - F.lag("total_population").over(w))
                / F.lag("total_population").over(w),
            ).otherwise(F.lit(0.0)).cast("double"),
        )

        demo_df = demo_df.select(
            "demo_year", "median_household_income", "total_population",
            "income_yoy_growth", "population_yoy_growth",
        )

        # Join on transaction_year
        df = df.join(
            broadcast(demo_df),
            F.col("transaction_year") == F.col("demo_year"),
            "left",
        )
        df = df.drop("demo_year")

        # Coalesce nulls
        for col_name in ["median_household_income", "total_population", "income_yoy_growth", "population_yoy_growth"]:
            df = df.withColumn(col_name, F.coalesce(F.col(col_name).cast("double"), F.lit(0.0)))

        # Derived: price_to_income_ratio = resale_price / median_household_income
        df = df.withColumn(
            "price_to_income_ratio",
            F.when(F.col("median_household_income") > 0, F.col("resale_price").cast("double") / F.col("median_household_income"))
             .otherwise(F.lit(0.0))
             .cast("double"),
        )

        created_features.extend([
            "median_household_income", "total_population",
            "price_to_income_ratio", "income_yoy_growth", "population_yoy_growth",
        ])
        logger.info("Demographic features created: median_household_income, total_population, price_to_income_ratio, income_yoy_growth, population_yoy_growth")

    # -----------------------------------------------------------------------
    # Interaction features from feature_config (no target leakage).
    # -----------------------------------------------------------------------
    interaction_specs = config.get("interaction_features", [])
    interaction_created = []
    skipped_interactions = []
    for spec in interaction_specs:
        kind = spec.get("kind")
        out_col = spec.get("name")
        ops = spec.get("operands", [])
        if not out_col or not kind or len(ops) != 2:
            continue
        a, b = ops[0], ops[1]
        if kind == "multiply":
            if a in df.columns and b in df.columns:
                df = df.withColumn(
                    out_col,
                    F.col(a).cast("double") * F.col(b).cast("double"),
                )
                interaction_created.append(out_col)
            else:
                skipped_interactions.append({"name": out_col, "reason": "missing operands on clean_df"})
        elif kind == "indexed_product":
            if a in df.columns and b in df.columns:
                df = df.withColumn(
                    out_col,
                    F.col(a).cast("double") * F.col(b).cast("double"),
                )
                interaction_created.append(out_col)
            else:
                skipped_interactions.append({"name": out_col, "reason": "needs encoded *_index columns"})

    feature_summary = {
        "features_created": created_features + interaction_created,
        "interaction_features": interaction_created,
        "skipped_interactions": skipped_interactions,
        "features_dropped": [],
        "final_feature_count": len(df.columns),
    }

    return {"feature_summary": feature_summary, "engineered_df": df}



## 6d. Node 3 — Train Models

Trains three PySpark MLlib regressors (LinearRegression, RandomForest, GBT) on the engineered DataFrame and computes RMSE, MAE, R-squared, and MAPE for each.

In [ ]:
# ---------------------------------------------------------------------------
# Node 3: train_models  (PySpark MLlib)
# ---------------------------------------------------------------------------

def train_models(state: AgentState) -> dict:
    """Train ML models using PySpark MLlib and compute evaluation metrics."""
    logger.info("Node: train_models")

    # TODO: implement PySpark MLlib training
    # 1. Train/test split: df.randomSplit([0.8, 0.2], seed=42)
    # 2. Assemble feature vector via VectorAssembler
    # 3. Train models:
    #    - pyspark.ml.regression.LinearRegression
    #    - pyspark.ml.regression.RandomForestRegressor (numTrees=100, maxDepth=10)
    #    - pyspark.ml.regression.GBTRegressor (maxIter=100, maxDepth=5)
    # 4. Evaluate with RegressionEvaluator (RMSE, MAE, R-squared)
    # 5. Optional: CrossValidator with ParamGridBuilder (k=5)

    model_results = {
        "LinearRegression": {"RMSE": 0.0, "MAE": 0.0, "R2": 0.0, "MAPE": 0.0},
        "RandomForest": {"RMSE": 0.0, "MAE": 0.0, "R2": 0.0, "MAPE": 0.0},
        "GBT": {"RMSE": 0.0, "MAE": 0.0, "R2": 0.0, "MAPE": 0.0},
    }

    return {"model_results": model_results}



## 6e. Node 4 — Evaluation Agent

LLM agent that receives model metrics and produces a comparative evaluation report: best model selection, reasoning, key findings, and recommendations.

In [ ]:
# ---------------------------------------------------------------------------
# Node 4: evaluation_agent
# ---------------------------------------------------------------------------

def evaluation_agent(state: AgentState) -> dict:
    """LLM agent that produces a comparative evaluation of model results."""
    logger.info("Node: evaluation_agent")

    # TODO: implement full agent skill
    # Expected: call OpenAI with model_results, return JSON with keys:
    #   best_model, reasoning, key_findings, recommendations, model_ranking

    evaluation_report = {
        "best_model": "",
        "reasoning": "",
        "key_findings": [],
        "recommendations": [],
        "model_ranking": [],
    }

    logs = list(state.get("agent_logs", []))
    logs.append(_log_entry("evaluation_agent", "Produced evaluation report (stub)"))

    return {"evaluation_report": evaluation_report, "agent_logs": logs}



## 6f. Pipeline Assembly and Execution

Builds the 4-node LangGraph `StateGraph` in linear sequence:

```
START → feature_engineering_agent → apply_feature_engineering → train_models → evaluation_agent → END
```

Visualises the graph and runs the pipeline on `clean_df`.

In [ ]:
# ---------------------------------------------------------------------------
# Pipeline builder
# ---------------------------------------------------------------------------

def build_pipeline():
    """Build and compile the LangGraph pipeline with 4 nodes in linear sequence."""
    graph = StateGraph(AgentState)

    # Add nodes
    graph.add_node("feature_engineering_agent", feature_engineering_agent)
    graph.add_node("apply_feature_engineering", apply_feature_engineering)
    graph.add_node("train_models", train_models)
    graph.add_node("evaluation_agent", evaluation_agent)

    # Linear edges: START -> feature_engineering_agent -> ... -> evaluation_agent -> END
    graph.add_edge(START, "feature_engineering_agent")
    graph.add_edge("feature_engineering_agent", "apply_feature_engineering")
    graph.add_edge("apply_feature_engineering", "train_models")
    graph.add_edge("train_models", "evaluation_agent")
    graph.add_edge("evaluation_agent", END)

    return graph.compile()


# ---------------------------------------------------------------------------
# Main: build, visualize, and run
# ---------------------------------------------------------------------------

pipeline = build_pipeline()

# Visualize the graph (works in Colab/Jupyter)
try:
    from IPython.display import Image, display
    display(Image(pipeline.get_graph().draw_mermaid_png()))
except Exception:
    pass

# Run the pipeline with clean_df metadata
initial_state = {
    "dataset_info": {},   # TODO: populate with actual schema/stats from clean_df
    "agent_logs": [],
    "engineered_df": None,
}

result = pipeline.invoke(initial_state)

print("=== Pipeline Complete ===")
print(f"Best model:  {result['evaluation_report']['best_model']}")
print(f"Reasoning:   {result['evaluation_report']['reasoning']}")
print(f"Agent logs:  {len(result['agent_logs'])} entries")

## 6g. Feature Validation & Spot-Check

Validates features produced by `apply_feature_engineering`:

**Distance features:**
- All four `dist_nearest_*` columns exist and are `DoubleType`, non-null, non-negative

**School quality features:**
- `dist_nearest_ip_school`, `dist_nearest_sap_school`, `dist_nearest_auto_school` (DoubleType, non-null)
- `has_ip_school_nearby`, `has_sap_school_nearby`, `has_affiliated_school_nearby` (IntegerType, binary 0/1)

**HDB Resale Price Index:**
- `hdb_rpi`, `price_to_rpi_ratio` (DoubleType, non-null)

**Population Demographics:**
- `median_household_income`, `total_population`, `price_to_income_ratio`, `income_yoy_growth`, `population_yoy_growth` (DoubleType, non-null)

**Correlation spot-check:** Pearson correlation of new features with `resale_price`

**Spot-check:** Bukit Merah flats should show ~0.3–1.0 km to nearest MRT (Redhill, Tiong Bahru, Queenstown stations nearby)

In [ ]:
from pyspark.sql import functions as F

engineered_df = result.get("engineered_df")
if engineered_df is None:
    raise ValueError("engineered_df missing from pipeline output. Ensure apply_feature_engineering returns it.")

# ===== Distance Feature Validation =====
required_distance_cols = [
    "dist_nearest_mrt",
    "dist_nearest_school",
    "dist_nearest_supermarket",
    "dist_nearest_hawker",
]

# --- AC: All distance columns exist ---
missing = [c for c in required_distance_cols if c not in engineered_df.columns]
assert not missing, f"Missing distance columns: {missing}"
print("PASS: All four distance columns present")

# --- AC: All distance columns are DoubleType ---
for c in required_distance_cols:
    dtype = dict(engineered_df.dtypes)[c]
    assert dtype == "double", f"{c} must be DoubleType, got {dtype}"
print("PASS: All distance columns are DoubleType")

# --- AC: No nulls, NaNs, or negative values ---
agg_exprs = []
for c in required_distance_cols:
    agg_exprs.extend([
        F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(f"{c}_nulls"),
        F.sum(F.when(F.isnan(F.col(c)), 1).otherwise(0)).alias(f"{c}_nans"),
        F.sum(F.when(F.col(c) < 0, 1).otherwise(0)).alias(f"{c}_negatives"),
    ])

quality_row = engineered_df.agg(*agg_exprs).collect()[0]
quality_issues = {k: v for k, v in quality_row.asDict().items() if v > 0}

if quality_issues:
    print("WARNING: Data quality issues found:", quality_issues)
else:
    print("PASS: All distance columns are non-null, non-NaN, and non-negative")

# ===== School Quality Feature Validation =====
print("\n=== School Quality Features ===")

school_quality_cols = {
    "dist_nearest_ip_school":       "double",
    "dist_nearest_sap_school":      "double",
    "dist_nearest_auto_school":     "double",
    "has_ip_school_nearby":         "int",
    "has_sap_school_nearby":        "int",
    "has_affiliated_school_nearby": "int",
}

missing_sq = [c for c in school_quality_cols if c not in engineered_df.columns]
assert not missing_sq, f"Missing school quality columns: {missing_sq}"
print("PASS: All school quality columns present")

for c, expected_type in school_quality_cols.items():
    dtype = dict(engineered_df.dtypes)[c]
    assert dtype == expected_type, f"{c} must be {expected_type}, got {dtype}"
print("PASS: All school quality columns have correct types")

sq_agg_exprs = []
for c in school_quality_cols:
    sq_agg_exprs.append(F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(f"{c}_nulls"))
sq_row = engineered_df.agg(*sq_agg_exprs).collect()[0]
sq_nulls = {k: v for k, v in sq_row.asDict().items() if v > 0}
assert not sq_nulls, f"School quality columns have nulls: {sq_nulls}"
print("PASS: All school quality columns are non-null")

# Binary columns should only contain 0 or 1
for bin_col in ["has_ip_school_nearby", "has_sap_school_nearby", "has_affiliated_school_nearby"]:
    invalid = engineered_df.filter(~F.col(bin_col).isin(0, 1)).count()
    assert invalid == 0, f"{bin_col} has values other than 0/1: {invalid} rows"
print("PASS: Binary school quality columns contain only 0/1 values")

# ===== HDB RPI Feature Validation =====
print("\n=== HDB Resale Price Index ===")

rpi_cols = {"hdb_rpi": "double", "price_to_rpi_ratio": "double"}

missing_rpi = [c for c in rpi_cols if c not in engineered_df.columns]
assert not missing_rpi, f"Missing RPI columns: {missing_rpi}"
print("PASS: All RPI columns present")

for c, expected_type in rpi_cols.items():
    dtype = dict(engineered_df.dtypes)[c]
    assert dtype == expected_type, f"{c} must be {expected_type}, got {dtype}"
print("PASS: All RPI columns are DoubleType")

rpi_agg = engineered_df.agg(
    F.sum(F.when(F.col("hdb_rpi").isNull(), 1).otherwise(0)).alias("hdb_rpi_nulls"),
    F.sum(F.when(F.col("price_to_rpi_ratio").isNull(), 1).otherwise(0)).alias("price_to_rpi_ratio_nulls"),
    F.round(F.avg("hdb_rpi"), 2).alias("avg_rpi"),
    F.round(F.avg("price_to_rpi_ratio"), 2).alias("avg_price_to_rpi"),
).collect()[0]

assert rpi_agg["hdb_rpi_nulls"] == 0, f"hdb_rpi has {rpi_agg['hdb_rpi_nulls']} nulls"
assert rpi_agg["price_to_rpi_ratio_nulls"] == 0, f"price_to_rpi_ratio has {rpi_agg['price_to_rpi_ratio_nulls']} nulls"
print(f"PASS: RPI columns non-null | avg hdb_rpi={rpi_agg['avg_rpi']}, avg price_to_rpi_ratio={rpi_agg['avg_price_to_rpi']}")

# ===== Demographics Feature Validation =====
print("\n=== Population Demographics ===")

demo_cols = {
    "median_household_income": "double",
    "total_population":        "double",
    "price_to_income_ratio":   "double",
    "income_yoy_growth":       "double",
    "population_yoy_growth":   "double",
}

missing_demo = [c for c in demo_cols if c not in engineered_df.columns]
assert not missing_demo, f"Missing demographic columns: {missing_demo}"
print("PASS: All demographic columns present")

for c, expected_type in demo_cols.items():
    dtype = dict(engineered_df.dtypes)[c]
    assert dtype == expected_type, f"{c} must be {expected_type}, got {dtype}"
print("PASS: All demographic columns are DoubleType")

demo_agg_exprs = []
for c in demo_cols:
    demo_agg_exprs.append(F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(f"{c}_nulls"))
demo_agg_exprs.extend([
    F.round(F.avg("median_household_income"), 2).alias("avg_income"),
    F.round(F.avg("total_population"), 0).alias("avg_pop"),
    F.round(F.avg("price_to_income_ratio"), 2).alias("avg_price_to_income"),
])
demo_row = engineered_df.agg(*demo_agg_exprs).collect()[0]

demo_nulls = {k: v for k, v in demo_row.asDict().items() if k.endswith("_nulls") and v > 0}
assert not demo_nulls, f"Demographic columns have nulls: {demo_nulls}"
print(f"PASS: Demographics non-null | avg income={demo_row['avg_income']}, avg pop={demo_row['avg_pop']}, avg price_to_income={demo_row['avg_price_to_income']}")

# ===== Correlation Spot-Check =====
print("\n=== Correlation Spot-Check ===")

# Compute Pearson correlation of new features with resale_price
corr_features = [
    "hdb_rpi", "price_to_rpi_ratio",
    "median_household_income", "total_population", "price_to_income_ratio",
    "dist_nearest_ip_school", "dist_nearest_sap_school", "dist_nearest_auto_school",
]
for feat in corr_features:
    if feat in engineered_df.columns:
        corr_val = engineered_df.stat.corr(feat, "resale_price")
        print(f"  corr({feat}, resale_price) = {corr_val:.4f}")

# --- AC: Spot-check — Bukit Merah flat distance features ---
print("\n=== Spot-Check: Bukit Merah Distance Features ===")
bukit_merah_df = engineered_df.filter(
    (F.lower(F.col("town")) == "bukit merah") &
    (~F.isnan("dist_nearest_mrt")) &
    (~F.isnan("dist_nearest_school")) &
    (~F.isnan("dist_nearest_supermarket")) &
    (~F.isnan("dist_nearest_hawker"))
)

bm_stats = bukit_merah_df.agg(
    F.round(F.min("dist_nearest_mrt"), 3).alias("min_mrt_km"),
    F.round(F.avg("dist_nearest_mrt"), 3).alias("avg_mrt_km"),
    F.round(F.max("dist_nearest_mrt"), 3).alias("max_mrt_km"),
    F.count("*").alias("n_flats"),
).collect()[0]

print(f"Bukit Merah flats: {bm_stats['n_flats']}")
print(f"dist_nearest_mrt — min: {bm_stats['min_mrt_km']} km, avg: {bm_stats['avg_mrt_km']} km, max: {bm_stats['max_mrt_km']} km")

# Show sample rows with all new features
bukit_merah_df.select(
    "town",
    "block",
    "flat_type",
    "dist_nearest_mrt",
    "dist_nearest_ip_school",
    "has_ip_school_nearby",
    "hdb_rpi",
    "median_household_income",
    "price_to_income_ratio",
).show(5, truncate=False)

## Cleanup

Stops the Spark session to release resources.

In [ ]:
spark.stop()
print('Spark session stopped.')